# XoFTR Image Matching Pipeline — DIMER E2E matching fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/xoftr-image-matching-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/xoftr-image-matching-pipeline/blob/main/tutorials/xoftr_image_matching_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-vismatch%2Fxoftr-ffcc4d?style=flat)](https://huggingface.co/vismatch/xoftr) [![Upstream](https://img.shields.io/badge/Upstream-OnderT%2FXoFTR-181717?style=flat&logo=github&logoColor=white)](https://github.com/OnderT/XoFTR) [![arXiv](https://img.shields.io/badge/arXiv-2404.09692-b31b1b.svg)](https://arxiv.org/abs/2404.09692)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** detector-free image matching (dense coarse-to-fine correspondences with sub-pixel refinement), homography-supervised evaluation and bounded supervised fine-tuning of the coarse transformer's last layers on a labelled pair dataset, using the pinned `vismatch/xoftr` weights in the vendored XoFTR network

**This notebook is standalone.** It carries the repository's package (7 modules under `src/xoftr_pipeline/`, at revision `26716c0cb9cc`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `d8ee7d89be3c9e5c157db3886db1c0f0e038b321` (~44 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `vismatch/xoftr` snapshot (a 44 MB `xoftr_640.safetensors`, loaded strictly into the carried network; no pickle is opened anywhere), fetches the 360 pinned iNaturalist photographs from the project's open-data bucket (about 39 MB, each refused on any byte-size or SHA-256 mismatch), cuts them per species into 216 / 48 / 96 training, validation and test photographs and turns each into a homography pair with an exact reference (two difficulty tiers), matches a drawn-shape pair through the inference contract with an input manifest and a rejection probe, measures the frozen matcher's precision at 3 px, inliers per pair and homography accuracy over the 96 test pairs beside the identity-guess and patch-nearest-neighbour baselines, runs a bounded fine-tuning of the coarse transformer's last two layers and coarse projection with the upstream coarse focal loss and validation-precision epoch selection, scores the held-out pairs again per tier, re-matches the drawn pair with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify match parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path took about 41 minutes on the build workstation after the downloads — dense matching at 640 px is heavy without a GPU (expect longer on a 2-vCPU hosted runtime); a CUDA runtime is used automatically when present and finishes in minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of JPEG / PNG photographs — at least eight, any subject, ideally textured — which are split by image, turned into homography pairs with the same seeded warps, and passed through the same validation, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the iNaturalist sample. The expected layout and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path. Real pairs of two different photographs of one scene need a reference homography or depth to be scored; the contract accepts only synthesised pairs with an exact `H`.

`vismatch/xoftr` hosts the checkpoints of XoFTR (Tuzcuoğlu, Köksal, Sofu, Kalkan and Alatan, CVPR 2024 Workshops): a detector-free, coarse-to-fine matcher in the LoFTR family — a ResNet backbone at 1/8 and 1/2 resolution, a linear-attention transformer over the coarse grid, dual-softmax coarse matching, a windowed fine stage and sub-pixel refinement — pre-trained for visible–thermal matching and released under the **Apache-2.0** licence; 11,091,722 parameters, 247 tensors. It returns a set of correspondences `(x0, y0) ↔ (x1, y1)` with a confidence each: the confidences are dual-softmax and fine-level scores, **not calibrated probabilities** that a match is right, they depend on the thresholds, and the matcher never abstains — any two images produce whatever passes the thresholds, overlapping or not. This repository carries the network as plain PyTorch (`modeling.py`, vendored from the upstream commit) so no third-party matching framework is installed.

What this notebook adds to inference is **adaptation with labelled pairs** — pairs whose correct answer is known exactly. The images are real: 360 CC0-licensed, research-grade iNaturalist photographs of six bird species (**CC0 1.0**; the fleet's SigLIP sample, 60 per species, one per observer), pinned by photo id, byte size and SHA-256 and fetched from the project's open-data bucket at run time. Each photograph becomes one pair with a seeded homography warp and seeded photometric changes, so every returned match has a **reprojection error** against the reference `H`. Two tiers alternate: `easy` (small perspective, ±10°, mild photometry) and `hard` (large perspective, ±35°, scale 0.6–1.4, strong photometry). The frozen matcher is already strong on this — the build record measured precision at 3 px of 0.925 over the 96 test pairs (0.866 on the hard tier) — so the honest question is narrow: does a bounded fine-tuning of the coarse transformer's last layers on 216 pairs move **precision at 3 px**, the **inlier count** and the **homography accuracy** on an image-disjoint test split, per tier, against two **non-neural baselines** (the **identity guess** and a **patch nearest neighbour**)? Nothing here is a quality claim about your images: it is one seeded split of one sample under synthetic warps.

**Snapshot note:** the pinned revision ships `xoftr_640.safetensors` (a 3-file manifest with the Hub card and the library marker) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the network is constructed; the 840-px sibling checkpoint hosted beside it is recorded in the card and not fetched.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot into a vendored network; build a labelled pair set with exact references from digest-pinned photographs, validate it and split it by photograph without leakage; match a drawn pair through the public API and read match confidences correctly (thresholded, uncalibrated, no abstention); measure the frozen matcher's precision, inlier count and homography accuracy beside two non-neural baselines and read the per-tier breakdown; run a bounded fine-tuning with the upstream coarse focal loss, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; re-match a drawing from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** object detection, semantic segmentation, OCR, caption generation, calibrated match probabilities or universal thresholds, geometric verification inside the pipeline (the evaluation's RANSAC is a metric, not a filter), pose or depth estimation, visible–thermal pairs (the checkpoint's own domain; the tutorial data are visible photographs), fine-tuning of the backbone, the positional encoding or the fine stage, training on images that are not the pinned sample or your own uploads, evaluation on HPatches, MegaDepth, VisTir or any benchmark proper (only one seeded 360-pair sample is scored here), and any claim that six bird species stand in for your scenes. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is slow: dense matching at 640 px costs about 2.5 s per pair on the build workstation, so the build record measured 300.4 s to match and score the 96 test pairs, 432.4 s for the two baselines (the patch search and RANSAC on ~3,400 matches per pair) and 1494.9 s for the 3 epochs of fine-tuning (216 pairs per epoch through the backbone and coarse transformer, the last two layers and the projection training, plus the per-epoch validation matching) — about 41 minutes in all on the build workstation with the snapshot and photographs already cached (a 2-vCPU hosted runtime will be slower still); a hosted T4 finishes the same path in minutes. The pinned `torch==2.14.0` install is the large download of the run; the checkpoint is 44 MB and the photographs about 39 MB.
- **Knowledge:** basic Python and PIL; what a homography is and why a warped copy of an image has an exact correspondence for every pixel; what precision, reprojection error and RANSAC measure and why none is a human judgement; why a high match confidence is not a correct match.
- **Data contract:** records are `{id, image0, image1, homography}` — two PIL images (or files decodable by Pillow) with sides in [64, 1024] px (the sample is built at 640 px on the long side, sides multiples of 8) and a finite, non-singular 3 × 3 reference mapping image0 pixels to image1 pixels. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 4..5,000 records; the sample is split by photograph (stratified per species) after pixel-digest de-duplication so no photograph lands in two splits. BYOD accepts one zip of photographs, from which the same seeded pairs are synthesised.
- **Validation is structural, not semantic:** every image is opened and decoded and every `H` checked for shape and rank, but nothing checks that `image1` really is `image0` under `H` — a wrong reference is scored without complaint, and a pair of unrelated photographs is matched without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches 360 JPEG/PNG files from `https://inaturalist-open-data.s3.amazonaws.com/photos/<id>/medium.<ext>` (about 39 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py` and refused on any mismatch; every photograph's iNaturalist observation page and observer login are kept in its record. Each photograph carries the CC0 1.0 licence its observer chose; nothing is committed to the repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `vismatch/xoftr` snapshot (~44 MB in total) at revision `d8ee7d89be3c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'torch==2.14.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'xoftr-image-matching-pipeline',
    'repository_revision': '26716c0cb9ccbce80a2be20cf28e379ffb30042b',
    'embedded_module': 'src/xoftr_pipeline/config.py',
    'embedded_modules': ['src/xoftr_pipeline/config.py', 'src/xoftr_pipeline/metrics.py', 'src/xoftr_pipeline/modeling.py', 'src/xoftr_pipeline/model.py', 'src/xoftr_pipeline/provenance.py', 'src/xoftr_pipeline/samples.py', 'src/xoftr_pipeline/pipeline.py'],
    'module_sha256': '48fee7606fb53c41bbe5477b39165172e1b245f8e15bf592709cc312a0ca6335',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/xoftr_pipeline/` @ `26716c0cb9cc`)

The next 7 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/7:** `src/xoftr_pipeline/config.py`

In [ ]:
from __future__ import annotations

# The XoFTR checkpoints as hosted by the vismatch (formerly image-matching-models) project on the
# Hub:
# plain safetensors state dicts of the upstream network with a `matcher.` prefix on every key.
MODEL_ID = "vismatch/xoftr"
MODEL_ID_FORMER = "image-matching-models/xoftr"  # the same files under the project's former name
MODEL_REVISION = "d8ee7d89be3c9e5c157db3886db1c0f0e038b321"
MODEL_LICENSE = "Apache-2.0"

# The served weight file: the 640-px training variant (the image-matching-models default).
MODEL_FILENAME = "xoftr_640.safetensors"
MODEL_SHA256 = "4d5ed62e8b41f862ecc5c660e31f1c450402966623d6a28e85acf7fbd794cc69"
MODEL_SIZE_BYTES = 44_419_304
# The 840-px sibling hosted beside it (recorded, not staged by default).
ALT_MODEL_FILENAME = "xoftr_840.safetensors"
ALT_MODEL_SHA256 = "3385e8d5121116805d99f700aaceddbbe9760a8dd22585e55404172e1ea0d488"
ALT_MODEL_SIZE_BYTES = 44_419_304
STATE_TENSORS = 247  # 231 float32 parameters + 16 int64 buffers (relative-position index tables)
PARAMETER_COUNT = 11_091_722

DEFAULT_MODEL_KEY = "xoftr"
UNSAFE_WEIGHT_EXTENSIONS = (
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)
ALLOWED_CHECKPOINT_FILES = (MODEL_FILENAME,)

# Inference contract.
COARSE_THRESHOLD = 0.3  # upstream MATCH_COARSE.THR
FINE_THRESHOLD = 0.1  # upstream FINE.THR
DIVISIBLE_BY = 8  # image sides are cropped down to a multiple of this (the 1/8 coarse grid)
MIN_SIDE = 64
MAX_SIDE = 1024

**Module 2/7:** `src/xoftr_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Homography-supervised matching metrics and two non-neural baselines, in numpy.

A record pairs an image with a warped copy of itself under a known 3 × 3 homography `H` (image0 → image1),
so every match has an exact reference: the reprojection error of `(x0, y0)` mapped by `H` against `(x1, y1)`.
For a set of records the pipeline reports:

- **precision at 3 px** (the fraction of returned matches with reprojection error under 3 px — the
  matching-precision reading), also at 1 px and 5 px;
- **matches per pair** and **inliers per pair** at 3 px (how much a downstream solver has to work with);
- **median reprojection error** of the inliers (sub-pixel accuracy);
- **homography accuracy at 3 px / 5 px**: the fraction of pairs whose homography, estimated from the
  matches by a normalised DLT inside a plain RANSAC loop, moves the four image corners by less than the
  threshold on average against the reference `H` (the usual HPatches-style reading; pairs with fewer than
  four inliers count as failures).

Two baselines a matcher must beat: the **identity guess** (every grid point maps to itself — correct only
where the warp is small) and a **patch nearest neighbour** (for each grid point of image0 the best
normalised-cross-correlation 15 × 15 patch of image1 within a search window — a classifier-free matcher
that knows the images through raw intensities). Both return the same match structure the model does and
are scored by the same code.
"""
# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
from PIL import Image

METRIC_DEFINITIONS = {
    "precision_3px": "fraction of returned matches whose reprojection error under the reference homography is below 3 px, averaged over pairs (a pair with no matches scores 0); in 0..1",
    "precision_1px": "the same at 1 px",
    "precision_5px": "the same at 5 px",
    "matches_per_pair": "mean number of returned matches per pair",
    "inliers_per_pair": "mean number of returned matches under 3 px per pair",
    "median_error_px": "median reprojection error of the inliers under 3 px, pooled over pairs; px",
    "homography_acc_3px": "fraction of pairs whose RANSAC-DLT homography from the matches moves the four corners by less than 3 px on average against the reference; in 0..1",
    "homography_acc_5px": "the same at 5 px",
}
THRESHOLDS = (1.0, 3.0, 5.0)
INLIER_PX = 3.0


def warp_points(points: np.ndarray, homography: np.ndarray) -> np.ndarray:
    """Apply a 3 × 3 homography to (N, 2) pixel coordinates."""
    pts = np.asarray(points, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError("points must be an (N, 2) array")
    hom = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ np.asarray(homography, dtype=np.float64).T
    with np.errstate(divide="ignore", invalid="ignore"):  # points at infinity under a degenerate candidate
        return hom[:, :2] / hom[:, 2:3]


def reprojection_errors(kpts0: np.ndarray, kpts1: np.ndarray, homography: np.ndarray) -> np.ndarray:
    """Per-match distance between `H · kpts0` and `kpts1`, in px."""
    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)
    if len(kpts0) != len(kpts1):
        raise ValueError("kpts0 and kpts1 must have the same length")
    if len(kpts0) == 0:
        return np.zeros((0,), dtype=np.float64)
    errors = np.linalg.norm(warp_points(kpts0, homography) - kpts1, axis=1)
    return np.where(np.isfinite(errors), errors, np.inf)


def _normalise(points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = points.mean(axis=0)
    scale = math.sqrt(2.0) / max(float(np.sqrt(((points - mean) ** 2).sum(axis=1)).mean()), 1e-9)
    transform = np.array([[scale, 0.0, -scale * mean[0]], [0.0, scale, -scale * mean[1]], [0.0, 0.0, 1.0]])
    hom = np.concatenate([points, np.ones((len(points), 1))], axis=1) @ transform.T
    return hom[:, :2], transform


def dlt_homography(kpts0: np.ndarray, kpts1: np.ndarray) -> np.ndarray | None:
    """Normalised direct linear transform from at least four correspondences; None when degenerate."""
    kpts0 = np.asarray(kpts0, dtype=np.float64)
    kpts1 = np.asarray(kpts1, dtype=np.float64)
    if len(kpts0) < 4:
        return None
    p0, t0 = _normalise(kpts0)
    p1, t1 = _normalise(kpts1)
    rows = []
    for (x, y), (u, v) in zip(p0, p1, strict=True):
        rows.append([-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u])
        rows.append([0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v])
    a = np.asarray(rows)
    try:
        _u, sigma, vt = np.linalg.svd(a)
    except np.linalg.LinAlgError:
        return None
    if sigma[-2] < 1e-12:  # rank-deficient: collinear points
        return None
    h_norm = vt[-1].reshape(3, 3)
    homography = np.linalg.inv(t1) @ h_norm @ t0
    if abs(homography[2, 2]) < 1e-12:
        return None
    return homography / homography[2, 2]


def ransac_homography(
    kpts0: np.ndarray,
    kpts1: np.ndarray,
    *,
    threshold: float = INLIER_PX,
    iterations: int = 500,
    seed: int = 0,
) -> tuple[np.ndarray | None, np.ndarray]:
    """A plain RANSAC over four-point DLT samples, refit on the consensus set. Returns (H or None, inlier mask)."""
    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)
    n = len(kpts0)
    if n < 4:
        return None, np.zeros((n,), dtype=bool)
    rng = np.random.default_rng(seed)
    best_mask = np.zeros((n,), dtype=bool)
    for _ in range(iterations):
        sample = rng.choice(n, size=4, replace=False)
        candidate = dlt_homography(kpts0[sample], kpts1[sample])
        if candidate is None:
            continue
        mask = reprojection_errors(kpts0, kpts1, candidate) < threshold
        if mask.sum() > best_mask.sum():
            best_mask = mask
            if best_mask.sum() == n:
                break
    if best_mask.sum() < 4:
        return None, best_mask
    refit = dlt_homography(kpts0[best_mask], kpts1[best_mask])
    if refit is None:
        return None, best_mask
    return refit, reprojection_errors(kpts0, kpts1, refit) < threshold


def corner_error(estimated: np.ndarray, reference: np.ndarray, size: tuple[int, int]) -> float:
    """Mean displacement of the four image corners between two homographies, in px."""
    width, height = size
    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])
    return float(np.linalg.norm(warp_points(corners, estimated) - warp_points(corners, reference), axis=1).mean())


def pair_metrics(match: Mapping[str, Any], homography: np.ndarray, size: tuple[int, int]) -> dict[str, Any]:
    """Per-pair scores for one match result `{kpts0, kpts1}` against the reference homography."""
    kpts0 = np.asarray(match["kpts0"], dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(match["kpts1"], dtype=np.float64).reshape(-1, 2)
    errors = reprojection_errors(kpts0, kpts1, homography)
    out: dict[str, Any] = {"n_matches": int(len(errors))}
    for t in THRESHOLDS:
        out[f"precision_{int(t)}px"] = float((errors < t).mean()) if len(errors) else 0.0
    inliers = errors[errors < INLIER_PX]
    out["n_inliers"] = int(len(inliers))
    out["inlier_errors"] = inliers.tolist()
    estimated, _mask = ransac_homography(kpts0, kpts1)
    out["corner_error_px"] = corner_error(estimated, homography, size) if estimated is not None else math.inf
    return out


def matching_metrics(per_pair: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Aggregate `pair_metrics` rows over a set of pairs."""
    if not per_pair:
        raise ValueError("at least one pair is required")
    pooled = [e for row in per_pair for e in row["inlier_errors"]]
    out: dict[str, Any] = {
        "n": len(per_pair),
        "matches_per_pair": float(np.mean([row["n_matches"] for row in per_pair])),
        "inliers_per_pair": float(np.mean([row["n_inliers"] for row in per_pair])),
        "median_error_px": float(np.median(pooled)) if pooled else math.inf,
        "homography_acc_3px": float(np.mean([row["corner_error_px"] < 3.0 for row in per_pair])),
        "homography_acc_5px": float(np.mean([row["corner_error_px"] < 5.0 for row in per_pair])),
        "definitions": METRIC_DEFINITIONS,
    }
    for t in THRESHOLDS:
        key = f"precision_{int(t)}px"
        out[key] = float(np.mean([row[key] for row in per_pair]))
    return out


# --------------------------------------------------------------------------------------------------
# non-neural baselines
# --------------------------------------------------------------------------------------------------


def grid_points(size: tuple[int, int], step: int = 32, margin: int = 16) -> np.ndarray:
    width, height = size
    xs = np.arange(margin, width - margin, step, dtype=np.float64)
    ys = np.arange(margin, height - margin, step, dtype=np.float64)
    gx, gy = np.meshgrid(xs, ys)
    return np.stack([gx.ravel(), gy.ravel()], axis=1)


def identity_baseline(image0: Image.Image, image1: Image.Image, *, step: int = 32) -> dict[str, Any]:
    """Every grid point of image0 is matched to the same coordinates in image1 (no motion assumed)."""
    pts = grid_points(image0.size, step=step)
    return {"kpts0": pts, "kpts1": pts.copy(), "confidence": np.ones(len(pts)), "baseline": "identity guess"}


def _gray(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("L"), dtype=np.float64)


def _ncc(patch: np.ndarray, window: np.ndarray) -> np.ndarray:
    """Normalised cross-correlation of a (p, p) patch over every (p, p) position of a (h, w) window."""
    p = patch.shape[0]
    h, w = window.shape
    if h < p or w < p:
        return np.zeros((0, 0))
    strides = np.lib.stride_tricks.sliding_window_view(window, (p, p))  # (h-p+1, w-p+1, p, p)
    tiles = strides.reshape(strides.shape[0], strides.shape[1], -1)
    tiles = tiles - tiles.mean(axis=2, keepdims=True)
    flat = (patch - patch.mean()).ravel()
    denom = np.sqrt((tiles**2).sum(axis=2) * (flat**2).sum()) + 1e-9
    return (tiles @ flat) / denom


def patch_neighbour_baseline(
    image0: Image.Image,
    image1: Image.Image,
    *,
    step: int = 32,
    patch: int = 15,
    search: int = 48,
) -> dict[str, Any]:
    """For each grid point of image0, the position in image1 (within ±`search` px) whose `patch` × `patch`
    neighbourhood has the highest normalised cross-correlation with the point's own patch."""
    g0, g1 = _gray(image0), _gray(image1)
    half = patch // 2
    pts0 = grid_points(image0.size, step=step, margin=max(16, half + 1))
    kpts0, kpts1, conf = [], [], []
    for x, y in pts0:
        xi, yi = int(round(x)), int(round(y))
        tile = g0[yi - half : yi + half + 1, xi - half : xi + half + 1]
        y0, y1 = max(0, yi - search - half), min(g1.shape[0], yi + search + half + 1)
        x0, x1 = max(0, xi - search - half), min(g1.shape[1], xi + search + half + 1)
        scores = _ncc(tile, g1[y0:y1, x0:x1])
        if scores.size == 0:
            continue
        best = np.unravel_index(int(np.argmax(scores)), scores.shape)
        kpts0.append([x, y])
        kpts1.append([x0 + best[1] + half, y0 + best[0] + half])
        conf.append(float(scores[best]))
    return {
        "kpts0": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),
        "kpts1": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),
        "confidence": np.asarray(conf, dtype=np.float64),
        "baseline": f"patch nearest neighbour ({patch}x{patch} NCC, ±{search} px search)",
    }

**Module 3/7:** `src/xoftr_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""XoFTR (Tuzcuoğlu, Köksal, Sofu, Kalkan and Alatan, CVPRW 2024) inference network, vendored from
https://github.com/OnderT/XoFTR at commit e0fbea431b30be9742effbf5577c90aa8eb938f9 (Apache-2.0):
``src/xoftr/backbone/resnet.py``, ``src/xoftr/utils/position_encoding.py`` and ``src/xoftr/xoftr_module/*``
concatenated in dependency order with the package-relative imports removed, plus the inference configuration
from ``src/config/default.py`` (``get_cfg_defaults(inference=True)`` lowered to a plain dict, as
image-matching-models builds it). No training utilities, datasets, Lightning or kornia code is carried; the
six ``einops.rearrange`` patterns upstream uses are provided by a local ``rearrange`` shim (plain ``reshape`` /
``permute``), so torch is the only dependency.

The state-dict key layout of the Hub checkpoints (``vismatch/xoftr``, formerly ``image-matching-models/xoftr``)
matches this module exactly: ``backbone.*``, ``pos_encoding.*``, ``loftr_coarse.*``, ``coarse_matching.*``,
``fine_process.*``, ``fine_matching.*``.
"""
# ruff: noqa: E501, N801, N802, N803, N806, E741, B905, F841, E712  -- vendored code kept as upstream wrote it, for auditability

from __future__ import annotations

import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Dropout, Module


def rearrange(tensor: torch.Tensor, pattern: str, **axes: int) -> torch.Tensor:
    """The six ``einops.rearrange`` patterns the upstream modules use, as plain ``view`` / ``permute`` so the
    vendored code reads exactly as upstream without an einops dependency. Any other pattern is refused."""
    if pattern == "b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c":
        return tensor.reshape(tensor.shape[0], axes["h0c"], axes["w0c"], axes["h1c"], axes["w1c"])
    if pattern == "b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)":
        return tensor.reshape(tensor.shape[0], axes["h0c"] * axes["w0c"], axes["h1c"] * axes["w1c"])
    if pattern == "n (h w) c -> n c h w":
        n, _, c = tensor.shape
        return tensor.reshape(n, axes["h"], axes["w"], c).permute(0, 3, 1, 2)
    if pattern == "n c h w -> n (h w) 1 c":
        return tensor.flatten(2).permute(0, 2, 1).unsqueeze(2)
    if pattern == "n (c ww) l -> n l ww c":
        n, cww, l = tensor.shape
        return tensor.reshape(n, cww // axes["ww"], axes["ww"], l).permute(0, 3, 2, 1)
    if pattern == "n c h w -> n (h w) c":
        return tensor.flatten(2).permute(0, 2, 1)
    raise ValueError(f"unsupported rearrange pattern: {pattern!r}")

UPSTREAM_REPOSITORY = "https://github.com/OnderT/XoFTR"
UPSTREAM_COMMIT = "e0fbea431b30be9742effbf5577c90aa8eb938f9"
RESOLUTION = (8, 2)


def default_config(*, coarse_thr: float = 0.3, fine_thr: float = 0.1, denser: bool = False) -> dict:
    """``get_cfg_defaults(inference=True)`` -> ``lower_config`` -> ``["xoftr"]`` from upstream ``src/config/default.py``,
    with the three knobs image-matching-models exposes."""
    return {
        "resolution": RESOLUTION,
        "fine_window_size": 5,
        "medium_window_size": 3,
        "resnet": {"initial_dim": 128, "block_dims": [128, 196, 256]},
        "coarse": {
            "inference": True,
            "d_model": 256,
            "d_ffn": 256,
            "nhead": 8,
            "layer_names": ["self", "cross"] * 4,
            "attention": "linear",
        },
        "match_coarse": {
            "inference": True,
            "d_model": 256,
            "thr": coarse_thr,
            "border_rm": 2,
            "match_type": "dual_softmax",
            "dsmax_temperature": 0.1,
            "train_coarse_percent": 0.2,
            "train_pad_num_gt_min": 200,
        },
        "fine": {
            "denser": denser,
            "inference": True,
            "dsmax_temperature": 0.1,
            "thr": fine_thr,
            "mlp_hidden_dim_coef": 2,
            "nhead_fine_level": 8,
            "nhead_medium_level": 7,
        },
        "loss": {
            "focal_alpha": 0.25,
            "focal_gamma": 2.0,
            "pos_weight": 1.0,
            "neg_weight": 1.0,
            "coarse_weight": 0.5,
            "fine_weight": 0.3,
            "sub_weight": 1 * 10**4,
        },
    }



# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/backbone/resnet.py
# ----------------------------------------------------------------------------------------------------

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution without padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False)


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)


class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.conv2 = conv3x3(planes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)

        if stride == 1:
            self.downsample = None
        else:
            self.downsample = nn.Sequential(
                conv1x1(in_planes, planes, stride=stride),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        y = x
        y = self.relu(self.bn1(self.conv1(y)))
        y = self.bn2(self.conv2(y))

        if self.downsample is not None:
            x = self.downsample(x)

        return self.relu(x+y)

class ResNet_8_2(nn.Module):
    """
    ResNet, output resolution are 1/8 and 1/2.
    Each block has 2 layers.
    """

    def __init__(self, config):
        super().__init__()
        # Config
        block = BasicBlock
        initial_dim = config['initial_dim']
        block_dims = config['block_dims']

        # Class Variable
        self.in_planes = initial_dim

        # Networks
        self.conv1 = nn.Conv2d(1, initial_dim, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(initial_dim)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(block, block_dims[0], stride=1)  # 1/2
        self.layer2 = self._make_layer(block, block_dims[1], stride=2)  # 1/4
        self.layer3 = self._make_layer(block, block_dims[2], stride=2)  # 1/8

        self.layer3_outconv = conv1x1(block_dims[2], block_dims[2])


        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, dim, stride=1):
        layer1 = block(self.in_planes, dim, stride=stride)
        layer2 = block(dim, dim, stride=1)
        layers = (layer1, layer2)

        self.in_planes = dim
        return nn.Sequential(*layers)

    def forward(self, x):
        # ResNet Backbone
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(x0)  # 1/2
        x2 = self.layer2(x1)  # 1/4
        x3 = self.layer3(x2)  # 1/8

        x3_out = self.layer3_outconv(x3)

        return x3_out, x2, x1


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/utils/position_encoding.py
# ----------------------------------------------------------------------------------------------------

class PositionEncodingSine(nn.Module):
    """
    This is a sinusoidal position encoding that generalized to 2-dimensional images
    """

    def __init__(self, d_model, max_shape=(256, 256)):
        """
        Args:
            max_shape (tuple): for 1/8 featmap, the max length of 256 corresponds to 2048 pixels
        """
        super().__init__()

        pe = torch.zeros((d_model, *max_shape))
        y_position = torch.ones(max_shape).cumsum(0).float().unsqueeze(0)
        x_position = torch.ones(max_shape).cumsum(1).float().unsqueeze(0)
        div_term = torch.exp(torch.arange(0, d_model//2, 2).float() * (-math.log(10000.0) / (d_model//2)))

        div_term = div_term[:, None, None]  # [C//4, 1, 1]
        pe[0::4, :, :] = torch.sin(x_position * div_term)
        pe[1::4, :, :] = torch.cos(x_position * div_term)
        pe[2::4, :, :] = torch.sin(y_position * div_term)
        pe[3::4, :, :] = torch.cos(y_position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0), persistent=False)  # [1, C, H, W]

    def forward(self, x):
        """
        Args:
            x: [N, C, H, W]
        """
        return x + self.pe[:, :, :x.size(2), :x.size(3)]


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr_module/linear_attention.py
# ----------------------------------------------------------------------------------------------------

"""
Linear Transformer proposed in "Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention"
Modified from: https://github.com/idiap/fast-transformers/blob/master/fast_transformers/attention/linear_attention.py
"""



def elu_feature_map(x):
    return torch.nn.functional.elu(x) + 1


class LinearAttention(Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.feature_map = elu_feature_map
        self.eps = eps

    def forward(self, queries, keys, values, q_mask=None, kv_mask=None):
        """ Multi-Head linear attention proposed in "Transformers are RNNs"
        Args:
            queries: [N, L, H, D]
            keys: [N, S, H, D]
            values: [N, S, H, D]
            q_mask: [N, L]
            kv_mask: [N, S]
        Returns:
            queried_values: (N, L, H, D)
        """
        Q = self.feature_map(queries)
        K = self.feature_map(keys)

        # set padded position to zero
        if q_mask is not None:
            Q = Q * q_mask[:, :, None, None]
        if kv_mask is not None:
            K = K * kv_mask[:, :, None, None]
            values = values * kv_mask[:, :, None, None]

        v_length = values.size(1)
        values = values / v_length  # prevent fp16 overflow
        KV = torch.einsum("nshd,nshv->nhdv", K, values)  # (S,D)' @ S,V
        Z = 1 / (torch.einsum("nlhd,nhd->nlh", Q, K.sum(dim=1)) + self.eps)
        queried_values = torch.einsum("nlhd,nhdv,nlh->nlhv", Q, KV, Z) * v_length

        return queried_values.contiguous()


class FullAttention(Module):
    def __init__(self, use_dropout=False, attention_dropout=0.1):
        super().__init__()
        self.use_dropout = use_dropout
        self.dropout = Dropout(attention_dropout)

    def forward(self, queries, keys, values, q_mask=None, kv_mask=None):
        """ Multi-head scaled dot-product attention, a.k.a full attention.
        Args:
            queries: [N, L, H, D]
            keys: [N, S, H, D]
            values: [N, S, H, D]
            q_mask: [N, L]
            kv_mask: [N, S]
        Returns:
            queried_values: (N, L, H, D)
        """

        # Compute the unnormalized attention and apply the masks
        QK = torch.einsum("nlhd,nshd->nlsh", queries, keys)
        if kv_mask is not None:
            QK.masked_fill_(~(q_mask[:, :, None, None] * kv_mask[:, None, :, None]), float('-inf'))

        # Compute the attention and the weighted average
        softmax_temp = 1. / queries.size(3)**.5  # sqrt(D)
        A = torch.softmax(softmax_temp * QK, dim=2)
        if self.use_dropout:
            A = self.dropout(A)

        queried_values = torch.einsum("nlsh,nshd->nlhd", A, values)

        return queried_values.contiguous()


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr_module/transformer.py
# ----------------------------------------------------------------------------------------------------

class LoFTREncoderLayer(nn.Module):
    def __init__(self,
                 d_model,
                 nhead,
                 attention='linear'):
        super().__init__()

        self.dim = d_model // nhead
        self.nhead = nhead

        # multi-head attention
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.attention = LinearAttention() if attention == 'linear' else FullAttention()
        self.merge = nn.Linear(d_model, d_model, bias=False)

        # feed-forward network
        self.mlp = nn.Sequential(
            nn.Linear(d_model*2, d_model*2, bias=False),
            nn.ReLU(True),
            nn.Linear(d_model*2, d_model, bias=False),
        )

        # norm and dropout
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, source, x_mask=None, source_mask=None):
        """
        Args:
            x (torch.Tensor): [N, L, C]
            source (torch.Tensor): [N, S, C]
            x_mask (torch.Tensor): [N, L] (optional)
            source_mask (torch.Tensor): [N, S] (optional)
        """
        bs = x.size(0)
        query, key, value = x, source, source

        # multi-head attention
        query = self.q_proj(query).view(bs, -1, self.nhead, self.dim)  # [N, L, (H, D)]
        key = self.k_proj(key).view(bs, -1, self.nhead, self.dim)  # [N, S, (H, D)]
        value = self.v_proj(value).view(bs, -1, self.nhead, self.dim)
        message = self.attention(query, key, value, q_mask=x_mask, kv_mask=source_mask)  # [N, L, (H, D)]
        message = self.merge(message.view(bs, -1, self.nhead*self.dim))  # [N, L, C]
        message = self.norm1(message)

        # feed-forward network
        message = self.mlp(torch.cat([x, message], dim=2))
        message = self.norm2(message)

        return x + message


class LocalFeatureTransformer(nn.Module):
    """A Local Feature Transformer (LoFTR) module."""

    def __init__(self, config):
        super().__init__()

        self.config = config
        self.d_model = config['d_model']
        self.nhead = config['nhead']
        self.layer_names = config['layer_names']
        encoder_layer = LoFTREncoderLayer(config['d_model'], config['nhead'], config['attention'])
        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer) for _ in range(len(self.layer_names))])
        self._reset_parameters()

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, feat0, feat1, mask0=None, mask1=None):
        """
        Args:
            feat0 (torch.Tensor): [N, L, C]
            feat1 (torch.Tensor): [N, S, C]
            mask0 (torch.Tensor): [N, L] (optional)
            mask1 (torch.Tensor): [N, S] (optional)
        """

        assert self.d_model == feat0.size(2), "the feature number of src and transformer must be equal"

        for layer, name in zip(self.layers, self.layer_names):
            if name == 'self':
                feat0 = layer(feat0, feat0, mask0, mask0)
                feat1 = layer(feat1, feat1, mask1, mask1)
            elif name == 'cross':
                feat0 = layer(feat0, feat1, mask0, mask1)
                feat1 = layer(feat1, feat0, mask1, mask0)
            else:
                raise KeyError

        return feat0, feat1


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr_module/coarse_matching.py
# ----------------------------------------------------------------------------------------------------

INF = 1e9

def mask_border(m, b: int, v):
    """ Mask borders with value
    Args:
        m (torch.Tensor): [N, H0, W0, H1, W1]
        b (int)
        v (m.dtype)
    """
    if b <= 0:
        return

    m[:, :b] = v
    m[:, :, :b] = v
    m[:, :, :, :b] = v
    m[:, :, :, :, :b] = v
    m[:, -b:] = v
    m[:, :, -b:] = v
    m[:, :, :, -b:] = v
    m[:, :, :, :, -b:] = v


def mask_border_with_padding(m, bd, v, p_m0, p_m1):
    if bd <= 0:
        return

    m[:, :bd] = v
    m[:, :, :bd] = v
    m[:, :, :, :bd] = v
    m[:, :, :, :, :bd] = v

    h0s, w0s = p_m0.sum(1).max(-1)[0].int(), p_m0.sum(-1).max(-1)[0].int()
    h1s, w1s = p_m1.sum(1).max(-1)[0].int(), p_m1.sum(-1).max(-1)[0].int()
    for b_idx, (h0, w0, h1, w1) in enumerate(zip(h0s, w0s, h1s, w1s)):
        m[b_idx, h0 - bd:] = v
        m[b_idx, :, w0 - bd:] = v
        m[b_idx, :, :, h1 - bd:] = v
        m[b_idx, :, :, :, w1 - bd:] = v


def compute_max_candidates(p_m0, p_m1):
    """Compute the max candidates of all pairs within a batch

    Args:
        p_m0, p_m1 (torch.Tensor): padded masks
    """
    h0s, w0s = p_m0.sum(1).max(-1)[0], p_m0.sum(-1).max(-1)[0]
    h1s, w1s = p_m1.sum(1).max(-1)[0], p_m1.sum(-1).max(-1)[0]
    max_cand = torch.sum(
        torch.min(torch.stack([h0s * w0s, h1s * w1s], -1), -1)[0])
    return max_cand


class CoarseMatching(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # general config
        d_model = config['d_model']
        self.thr = config['thr']
        self.inference = config['inference']
        self.border_rm = config['border_rm']
        # -- # for trainig fine-level XoFTR
        self.train_coarse_percent = config['train_coarse_percent']
        self.train_pad_num_gt_min = config['train_pad_num_gt_min']
        self.final_proj = nn.Linear(d_model, d_model, bias=True)

        self.temperature = config['dsmax_temperature']

    def forward(self, feat_c0, feat_c1, data, mask_c0=None, mask_c1=None):
        """
        Args:
            feat0 (torch.Tensor): [N, L, C]
            feat1 (torch.Tensor): [N, S, C]
            data (dict)
            mask_c0 (torch.Tensor): [N, L] (optional)
            mask_c1 (torch.Tensor): [N, S] (optional)
        Update:
            data (dict): {
                'b_ids' (torch.Tensor): [M'],
                'i_ids' (torch.Tensor): [M'],
                'j_ids' (torch.Tensor): [M'],
                'gt_mask' (torch.Tensor): [M'],
                'mkpts0_c' (torch.Tensor): [M, 2],
                'mkpts1_c' (torch.Tensor): [M, 2],
                'mconf' (torch.Tensor): [M]}
            NOTE: M' != M during training.
        """

        feat_c0 = self.final_proj(feat_c0)
        feat_c1 = self.final_proj(feat_c1)

        # normalize
        feat_c0, feat_c1 = map(lambda feat: feat / feat.shape[-1]**.5,
                               [feat_c0, feat_c1])

        sim_matrix = torch.einsum("nlc,nsc->nls", feat_c0,
                                    feat_c1) / self.temperature
        if mask_c0 is not None:
            sim_matrix.masked_fill_(
                ~(mask_c0[..., None] * mask_c1[:, None]).bool(),
                -INF)
        if self.inference:
            # predict coarse matches from conf_matrix
            data.update(**self.get_coarse_match_inference(sim_matrix, data))
        else:
            conf_matrix_0_to_1 = F.softmax(sim_matrix, 2)
            conf_matrix_1_to_0 = F.softmax(sim_matrix, 1)
            data.update({'conf_matrix_0_to_1': conf_matrix_0_to_1,
                        'conf_matrix_1_to_0': conf_matrix_1_to_0
                        })
            # predict coarse matches from conf_matrix
            data.update(**self.get_coarse_match_training(conf_matrix_0_to_1, conf_matrix_1_to_0, data))

    @torch.no_grad()
    def get_coarse_match_training(self, conf_matrix_0_to_1, conf_matrix_1_to_0, data):
        """
        Args:
            conf_matrix_0_to_1 (torch.Tensor): [N, L, S]
            conf_matrix_1_to_0 (torch.Tensor): [N, L, S]
            data (dict): with keys ['hw0_i', 'hw1_i', 'hw0_c', 'hw1_c']
        Returns:
            coarse_matches (dict): {
                'b_ids' (torch.Tensor): [M'],
                'i_ids' (torch.Tensor): [M'],
                'j_ids' (torch.Tensor): [M'],
                'gt_mask' (torch.Tensor): [M'],
                'm_bids' (torch.Tensor): [M],
                'mkpts0_c' (torch.Tensor): [M, 2],
                'mkpts1_c' (torch.Tensor): [M, 2],
                'mconf' (torch.Tensor): [M]}
        """
        axes_lengths = {
            'h0c': data['hw0_c'][0],
            'w0c': data['hw0_c'][1],
            'h1c': data['hw1_c'][0],
            'w1c': data['hw1_c'][1]
        }
        _device = conf_matrix_0_to_1.device

        # confidence thresholding
        # {(nearest neighbour for 0 to 1) U (nearest neighbour for 1 to 0)}
        mask = torch.logical_or((conf_matrix_0_to_1 > self.thr) * (conf_matrix_0_to_1 == conf_matrix_0_to_1.max(dim=2, keepdim=True)[0]),
                               (conf_matrix_1_to_0 > self.thr) * (conf_matrix_1_to_0 == conf_matrix_1_to_0.max(dim=1, keepdim=True)[0]))

        mask = rearrange(mask, 'b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c',
                         **axes_lengths)
        if 'mask0' not in data:
            mask_border(mask, self.border_rm, False)
        else:
            mask_border_with_padding(mask, self.border_rm, False,
                                     data['mask0'], data['mask1'])
        mask = rearrange(mask, 'b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)',
                         **axes_lengths)

        # find all valid coarse matches
        b_ids, i_ids, j_ids = mask.nonzero(as_tuple=True)

        mconf = torch.maximum(conf_matrix_0_to_1[b_ids, i_ids, j_ids], conf_matrix_1_to_0[b_ids, i_ids, j_ids])

        # random sampling of training samples for fine-level XoFTR
        # (optional) pad samples with gt coarse-level matches
        if self.training:
            # NOTE:
            # the sampling is performed across all pairs in a batch without manually balancing
            # samples for fine-level increases w.r.t. batch_size
            if 'mask0' not in data:
                num_candidates_max = mask.size(0) * max(
                    mask.size(1), mask.size(2))
            else:
                num_candidates_max = compute_max_candidates(
                    data['mask0'], data['mask1'])
            num_matches_train = int(num_candidates_max *
                                    self.train_coarse_percent)
            num_matches_pred = len(b_ids)
            assert self.train_pad_num_gt_min < num_matches_train, "min-num-gt-pad should be less than num-train-matches"

            # pred_indices is to select from prediction
            if num_matches_pred <= num_matches_train - self.train_pad_num_gt_min:
                pred_indices = torch.arange(num_matches_pred, device=_device)
            else:
                pred_indices = torch.randint(
                    num_matches_pred,
                    (num_matches_train - self.train_pad_num_gt_min, ),
                    device=_device)

            # gt_pad_indices is to select from gt padding. e.g. max(3787-4800, 200)
            gt_pad_indices = torch.randint(
                    len(data['spv_b_ids']),
                    (max(num_matches_train - num_matches_pred,
                        self.train_pad_num_gt_min), ),
                    device=_device)
            mconf_gt = torch.zeros(len(data['spv_b_ids']), device=_device)  # set conf of gt paddings to all zero

            b_ids, i_ids, j_ids, mconf = map(
                lambda x, y: torch.cat([x[pred_indices], y[gt_pad_indices]],
                                       dim=0),
                *zip([b_ids, data['spv_b_ids']], [i_ids, data['spv_i_ids']],
                     [j_ids, data['spv_j_ids']], [mconf, mconf_gt]))

        # these matches are selected patches that feed into fine-level network
        coarse_matches = {'b_ids': b_ids, 'i_ids': i_ids, 'j_ids': j_ids}

        # update with matches in original image resolution
        scale = data['hw0_i'][0] / data['hw0_c'][0]
        scale0 = scale * data['scale0'][b_ids] if 'scale0' in data else scale
        scale1 = scale * data['scale1'][b_ids] if 'scale1' in data else scale
        mkpts0_c = torch.stack(
            [i_ids % data['hw0_c'][1], torch.div(i_ids, data['hw0_c'][1], rounding_mode='trunc')],
            dim=1) * scale0
        mkpts1_c = torch.stack(
            [j_ids % data['hw1_c'][1], torch.div(j_ids, data['hw1_c'][1], rounding_mode='trunc')],
            dim=1) * scale1

        # these matches is the current prediction (for visualization)
        coarse_matches.update({
            'gt_mask': mconf == 0,
            'm_bids': b_ids[mconf != 0],  # mconf == 0 => gt matches
            'mkpts0_c': mkpts0_c[mconf != 0],
            'mkpts1_c': mkpts1_c[mconf != 0],
            'mconf': mconf[mconf != 0]
        })

        return coarse_matches

    @torch.no_grad()
    def get_coarse_match_inference(self, sim_matrix, data):
        """
        Args:
            sim_matrix (torch.Tensor): [N, L, S]
            data (dict): with keys ['hw0_i', 'hw1_i', 'hw0_c', 'hw1_c']
        Returns:
            coarse_matches (dict): {
                'b_ids' (torch.Tensor): [M'],
                'i_ids' (torch.Tensor): [M'],
                'j_ids' (torch.Tensor): [M'],
                'gt_mask' (torch.Tensor): [M'],
                'm_bids' (torch.Tensor): [M],
                'mkpts0_c' (torch.Tensor): [M, 2],
                'mkpts1_c' (torch.Tensor): [M, 2],
                'mconf' (torch.Tensor): [M]}
        """
        axes_lengths = {
            'h0c': data['hw0_c'][0],
            'w0c': data['hw0_c'][1],
            'h1c': data['hw1_c'][0],
            'w1c': data['hw1_c'][1]
        }

        # softmax for 0 to 1
        conf_matrix_ = F.softmax(sim_matrix, 2)

        # confidence thresholding and nearest neighbour for 0 to 1
        mask = (conf_matrix_ > self.thr) * (conf_matrix_ == conf_matrix_.max(dim=2, keepdim=True)[0])

        # unlike training, reuse the same conf martix to decrease the vram consumption
        # softmax for 0 to 1
        conf_matrix_ = F.softmax(sim_matrix, 1)

        # update mask {(nearest neighbour for 0 to 1) U (nearest neighbour for 1 to 0)}
        mask = torch.logical_or(mask,
                                 (conf_matrix_ > self.thr) * (conf_matrix_ == conf_matrix_.max(dim=1, keepdim=True)[0]))

        mask = rearrange(mask, 'b (h0c w0c) (h1c w1c) -> b h0c w0c h1c w1c',
                    **axes_lengths)
        if 'mask0' not in data:
            mask_border(mask, self.border_rm, False)
        else:
            mask_border_with_padding(mask, self.border_rm, False,
                                     data['mask0'], data['mask1'])
        mask = rearrange(mask, 'b h0c w0c h1c w1c -> b (h0c w0c) (h1c w1c)',
                         **axes_lengths)

        # find all valid coarse matches
        b_ids, i_ids, j_ids = mask.nonzero(as_tuple=True)

        # mconf = torch.maximum(conf_matrix_0_to_1[b_ids, i_ids, j_ids], conf_matrix_1_to_0[b_ids, i_ids, j_ids])

        # these matches are selected patches that feed into fine-level network
        coarse_matches = {'b_ids': b_ids, 'i_ids': i_ids, 'j_ids': j_ids}

        # update with matches in original image resolution
        scale = data['hw0_i'][0] / data['hw0_c'][0]
        scale0 = scale * data['scale0'][b_ids] if 'scale0' in data else scale
        scale1 = scale * data['scale1'][b_ids] if 'scale1' in data else scale
        mkpts0_c = torch.stack(
            [i_ids % data['hw0_c'][1], torch.div(i_ids, data['hw0_c'][1], rounding_mode='trunc')],
            dim=1) * scale0
        mkpts1_c = torch.stack(
            [j_ids % data['hw1_c'][1], torch.div(j_ids, data['hw1_c'][1], rounding_mode='trunc')],
            dim=1) * scale1

        # these matches are the current coarse level predictions
        coarse_matches.update({
            'm_bids': b_ids,  # mconf == 0 => gt matches
            'mkpts0_c': mkpts0_c,
            'mkpts1_c': mkpts1_c,
        })

        return coarse_matches


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr_module/fine_process.py
# ----------------------------------------------------------------------------------------------------

class Mlp(nn.Module):
    """Multi-Layer Perceptron (MLP)"""

    def __init__(self,
                 in_dim,
                 hidden_dim=None,
                 out_dim=None,
                 act_layer=nn.GELU):
        """
        Args:
            in_dim: input features dimension
            hidden_dim: hidden features dimension
            out_dim: output features dimension
            act_layer: activation function
        """
        super().__init__()
        out_dim = out_dim or in_dim
        hidden_dim = hidden_dim or in_dim
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.out_dim = out_dim

    def forward(self, x):
        x_size = x.size()
        x = x.view(-1, x_size[-1])
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = x.view(*x_size[:-1], self.out_dim)
        return x


class VanillaAttention(nn.Module):
    def __init__(self,
                 dim,
                 num_heads=8,
                 proj_bias=False):
        super().__init__()
        """
        Args:
            dim: feature dimension
            num_heads: number of attention head
            proj_bias: bool use query, key, value bias
        """
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.softmax_temp = self.head_dim ** -0.5
        self.kv_proj = nn.Linear(dim, dim * 2, bias=proj_bias)
        self.q_proj = nn.Linear(dim, dim, bias=proj_bias)
        self.merge = nn.Linear(dim, dim)

    def forward(self, x_q, x_kv=None):
        """
        Args:
            x_q (torch.Tensor): [N, L, C]
            x_kv (torch.Tensor): [N, S, C]
        """
        if x_kv is None:
            x_kv = x_q
        bs, _, dim = x_q.shape
        bs, _, dim = x_kv.shape
        # [N, S, 2, H, D] => [2, N, H, S, D]
        kv = self.kv_proj(x_kv).reshape(bs, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # [N, L, H, D] => [N, H, L, D]
        q = self.q_proj(x_q).reshape(bs, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k, v = kv[0].transpose(-2, -1).contiguous(), kv[1].contiguous() # [N, H, D, S], [N, H, S, D]
        attn = (q @ k) * self.softmax_temp # [N, H, L, S]
        attn = attn.softmax(dim=-1)
        x_q = (attn @ v).transpose(1, 2).reshape(bs, -1, dim)
        x_q = self.merge(x_q)
        return x_q


class CrossBidirectionalAttention(nn.Module):
    def __init__(self, dim, num_heads, proj_bias = False):
        super().__init__()
        """
        Args:
            dim: feature dimension
            num_heads: number of attention head
            proj_bias: bool use query, key, value bias
        """

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.softmax_temp = self.head_dim ** -0.5
        self.qk_proj = nn.Linear(dim, dim, bias=proj_bias)
        self.v_proj = nn.Linear(dim, dim, bias=proj_bias)
        self.merge = nn.Linear(dim, dim, bias=proj_bias)
        self.temperature = nn.Parameter(torch.tensor([0.0]), requires_grad=True)
        # print(self.temperature)

    def map_(self, func, x0, x1):
        return func(x0), func(x1)

    def forward(self, x0, x1):
        """
        Args:
            x0 (torch.Tensor): [N, L, C]
            x1 (torch.Tensor): [N, S, C]
        """
        bs = x0.size(0)

        qk0, qk1 = self.map_(self.qk_proj, x0, x1)
        v0, v1 = self.map_(self.v_proj, x0, x1)
        qk0, qk1, v0, v1 = map(
            lambda t: t.reshape(bs, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3).contiguous(),
            (qk0, qk1, v0, v1))

        qk0, qk1 = qk0 * self.softmax_temp**0.5, qk1 * self.softmax_temp**0.5
        sim = qk0 @ qk1.transpose(-2,-1).contiguous()
        attn01 = F.softmax(sim, dim=-1)
        attn10 = F.softmax(sim.transpose(-2, -1).contiguous(), dim=-1)
        x0 = attn01 @ v1
        x1 = attn10 @ v0
        x0, x1 = self.map_(lambda t: t.transpose(1, 2).flatten(start_dim=-2),
                        x0, x1)
        x0, x1 = self.map_(self.merge, x0, x1)

        return x0, x1


class SwinPosEmbMLP(nn.Module):
    def __init__(self,
                 dim):
        super().__init__()
        self.pos_embed = None
        self.pos_mlp = nn.Sequential(nn.Linear(2, 512, bias=True),
                                        nn.ReLU(),
                                        nn.Linear(512, dim, bias=False))

    def forward(self, x):
        seq_length = x.shape[1]
        if self.pos_embed is None or self.training:
            seq_length = int(seq_length**0.5)
            coords = torch.arange(0, seq_length, device=x.device, dtype = x.dtype)
            grid = torch.stack(torch.meshgrid([coords, coords])).contiguous().unsqueeze(0)
            grid -= seq_length // 2
            grid /= (seq_length // 2)
            self.pos_embed = self.pos_mlp(grid.flatten(2).transpose(1,2))
        x = x + self.pos_embed
        return x


class WindowSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, mlp_hidden_coef, use_pre_pos_embed=False):
        super().__init__()
        self.mlp = Mlp(in_dim=dim*2, hidden_dim=dim*mlp_hidden_coef, out_dim=dim, act_layer=nn.GELU)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.attn = VanillaAttention(dim, num_heads=num_heads)
        self.pos_embed = SwinPosEmbMLP(dim)
        self.pos_embed_pre = SwinPosEmbMLP(dim) if use_pre_pos_embed else nn.Identity()

    def forward(self, x, x_pre):
        ww = x.shape[1]
        ww_pre = x_pre.shape[1]
        x = self.pos_embed(x)
        x_pre = self.pos_embed_pre(x_pre)
        x = torch.cat((x, x_pre), dim=1)
        x = x + self.gamma*self.norm1(self.mlp(torch.cat([x, self.attn(self.norm2(x))], dim=-1)))
        x, x_pre = x.split([ww, ww_pre], dim=1)
        return x, x_pre


class WindowCrossAttention(nn.Module):
    def __init__(self, dim, num_heads, mlp_hidden_coef):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_dim=dim*2, hidden_dim=dim*mlp_hidden_coef, out_dim=dim, act_layer=nn.GELU)
        self.cross_attn = CrossBidirectionalAttention(dim, num_heads=num_heads, proj_bias=False)
        self.gamma = nn.Parameter(torch.ones(dim))

    def forward(self, x0, x1):
        m_x0, m_x1 = self.cross_attn(self.norm1(x0), self.norm1(x1))
        x0 = x0 + self.gamma*self.norm2(self.mlp(torch.cat([x0, m_x0], dim=-1)))
        x1 = x1 + self.gamma*self.norm2(self.mlp(torch.cat([x1, m_x1], dim=-1)))
        return x0, x1


class FineProcess(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Config
        block_dims = config['resnet']['block_dims']
        self.block_dims = block_dims
        self.W_f = config['fine_window_size']
        self.W_m = config['medium_window_size']
        nhead_f = config["fine"]['nhead_fine_level']
        nhead_m = config["fine"]['nhead_medium_level']
        mlp_hidden_coef = config["fine"]['mlp_hidden_dim_coef']

        # Networks
        self.conv_merge = nn.Sequential(nn.Conv2d(block_dims[2]*2, block_dims[1], kernel_size=1, stride=1, padding=0, bias=False),
                                        nn.Conv2d(block_dims[1], block_dims[1], kernel_size=3, stride=1, padding=1, groups=block_dims[1], bias=False),
                                        nn.BatchNorm2d(block_dims[1])
                                        )
        self.out_conv_m = nn.Conv2d(block_dims[1], block_dims[1], kernel_size=1, stride=1, padding=0, bias=False)
        self.out_conv_f = nn.Conv2d(block_dims[0], block_dims[0], kernel_size=1, stride=1, padding=0, bias=False)
        self.self_attn_m = WindowSelfAttention(block_dims[1], num_heads=nhead_m,
                                                mlp_hidden_coef=mlp_hidden_coef, use_pre_pos_embed=False)
        self.cross_attn_m = WindowCrossAttention(block_dims[1], num_heads=nhead_m,
                                                  mlp_hidden_coef=mlp_hidden_coef)
        self.self_attn_f = WindowSelfAttention(block_dims[0], num_heads=nhead_f,
                                                mlp_hidden_coef=mlp_hidden_coef, use_pre_pos_embed=True)
        self.cross_attn_f = WindowCrossAttention(block_dims[0], num_heads=nhead_f,
                                                  mlp_hidden_coef=mlp_hidden_coef)
        self.down_proj_m_f = nn.Linear(block_dims[1], block_dims[0], bias=False)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def pre_process(self, feat_f0, feat_f1, feat_m0, feat_m1, feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data):
        W_f = self.W_f
        W_m = self.W_m
        data.update({'W_f': W_f,
                'W_m': W_m})

        # merge coarse features before and after loftr layer, and down proj channel dimesions
        feat_c0 = rearrange(feat_c0, 'n (h w) c -> n c h w', h =data["hw0_c"][0], w =data["hw0_c"][1])
        feat_c1 = rearrange(feat_c1, 'n (h w) c -> n c h w', h =data["hw1_c"][0], w =data["hw1_c"][1])
        feat_c0 = self.conv_merge(torch.cat([feat_c0, feat_c0_pre], dim=1))
        feat_c1 = self.conv_merge(torch.cat([feat_c1, feat_c1_pre], dim=1))
        feat_c0 = rearrange(feat_c0, 'n c h w -> n (h w) 1 c')
        feat_c1 = rearrange(feat_c1, 'n c h w -> n (h w) 1 c')

        stride_f = data['hw0_f'][0] // data['hw0_c'][0]
        stride_m = data['hw0_m'][0] // data['hw0_c'][0]

        if feat_m0.shape[2] == feat_m1.shape[2] and feat_m0.shape[3] == feat_m1.shape[3]:
            feat_m = self.out_conv_m(torch.cat([feat_m0, feat_m1], dim=0))
            feat_m0, feat_m1 = torch.chunk(feat_m, 2, dim=0)
            feat_f = self.out_conv_f(torch.cat([feat_f0, feat_f1], dim=0))
            feat_f0, feat_f1 = torch.chunk(feat_f, 2, dim=0)
        else:
            feat_m0 = self.out_conv_m(feat_m0)
            feat_m1 = self.out_conv_m(feat_m1)
            feat_f0 = self.out_conv_f(feat_f0)
            feat_f1 = self.out_conv_f(feat_f1)

        # 1. unfold (crop windows) all local windows
        feat_m0_unfold = F.unfold(feat_m0, kernel_size=(W_m, W_m), stride=stride_m, padding=W_m//2)
        feat_m0_unfold = rearrange(feat_m0_unfold, 'n (c ww) l -> n l ww c', ww=W_m**2)
        feat_m1_unfold = F.unfold(feat_m1, kernel_size=(W_m, W_m), stride=stride_m, padding=W_m//2)
        feat_m1_unfold = rearrange(feat_m1_unfold, 'n (c ww) l -> n l ww c', ww=W_m**2)

        feat_f0_unfold = F.unfold(feat_f0, kernel_size=(W_f, W_f), stride=stride_f, padding=W_f//2)
        feat_f0_unfold = rearrange(feat_f0_unfold, 'n (c ww) l -> n l ww c', ww=W_f**2)
        feat_f1_unfold = F.unfold(feat_f1, kernel_size=(W_f, W_f), stride=stride_f, padding=W_f//2)
        feat_f1_unfold = rearrange(feat_f1_unfold, 'n (c ww) l -> n l ww c', ww=W_f**2)

        # 2. select only the predicted matches
        feat_c0 = feat_c0[data['b_ids'], data['i_ids']] # [n, ww, cm]
        feat_c1 = feat_c1[data['b_ids'], data['j_ids']]

        feat_m0_unfold = feat_m0_unfold[data['b_ids'], data['i_ids']]  # [n, ww, cm]
        feat_m1_unfold = feat_m1_unfold[data['b_ids'], data['j_ids']]

        feat_f0_unfold = feat_f0_unfold[data['b_ids'], data['i_ids']]  # [n, ww, cf]
        feat_f1_unfold = feat_f1_unfold[data['b_ids'], data['j_ids']]

        return feat_c0, feat_c1, feat_m0_unfold, feat_m1_unfold, feat_f0_unfold, feat_f1_unfold

    def forward(self, feat_f0, feat_f1, feat_m0, feat_m1, feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data):
        """
        Args:
            feat_f0 (torch.Tensor): [N, C, H, W]
            feat_f1 (torch.Tensor): [N, C, H, W]
            feat_m0 (torch.Tensor): [N, C, H, W]
            feat_m1 (torch.Tensor): [N, C, H, W]
            feat_c0 (torch.Tensor): [N, L, C]
            feat_c1 (torch.Tensor): [N, S, C]
            feat_c0_pre (torch.Tensor): [N, C, H, W]
            feat_c1_pre (torch.Tensor): [N, C, H, W]
            data (dict): with keys ['hw0_c', 'hw1_c', 'hw0_m', 'hw1_m', 'hw0_f', 'hw1_f', 'b_ids', 'j_ids']
        """

        # upstream note: "Check for this case" (kept as written; a to-do marker in the vendored source)
        if data['b_ids'].shape[0] == 0:
            feat0 = torch.empty(0, self.W_f**2, self.block_dims[0], device=feat_f0.device)
            feat1 = torch.empty(0, self.W_f**2, self.block_dims[0], device=feat_f0.device)
            return feat0, feat1

        feat_c0, feat_c1, feat_m0_unfold, feat_m1_unfold, \
            feat_f0_unfold, feat_f1_unfold = self.pre_process(feat_f0, feat_f1, feat_m0, feat_m1,
                                                               feat_c0, feat_c1, feat_c0_pre, feat_c1_pre, data)

        # self attention (c + m)
        feat_m_unfold, _ = self.self_attn_m(torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0),
                                                         torch.cat([feat_c0, feat_c1], dim=0))
        feat_m0_unfold, feat_m1_unfold = torch.chunk(feat_m_unfold, 2, dim=0)

        # cross attention (m0 <-> m1)
        feat_m0_unfold, feat_m1_unfold = self.cross_attn_m(feat_m0_unfold, feat_m1_unfold)

        # down proj m
        feat_m_unfold = self.down_proj_m_f(torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0))
        feat_m0_unfold, feat_m1_unfold = torch.chunk(feat_m_unfold, 2, dim=0)

        # self attention (m + f)
        feat_f_unfold, _ = self.self_attn_f(torch.cat([feat_f0_unfold, feat_f1_unfold], dim=0),
                                                         torch.cat([feat_m0_unfold, feat_m1_unfold], dim=0))
        feat_f0_unfold, feat_f1_unfold = torch.chunk(feat_f_unfold, 2, dim=0)

        # cross attention (f0 <-> f1)
        feat_f0_unfold, feat_f1_unfold = self.cross_attn_f(feat_f0_unfold, feat_f1_unfold)

        return feat_f0_unfold, feat_f1_unfold


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr_module/fine_matching.py
# ----------------------------------------------------------------------------------------------------

class FineSubMatching(nn.Module):
    """Fine-level and Sub-pixel matching"""

    def __init__(self, config):
        super().__init__()
        self.temperature = config['fine']['dsmax_temperature']
        self.W_f = config['fine_window_size']
        self.denser = config['fine']['denser']
        self.inference = config['fine']['inference']
        dim_f = config['resnet']['block_dims'][0]
        self.fine_thr = config['fine']['thr']
        self.fine_proj = nn.Linear(dim_f, dim_f, bias=False)
        self.subpixel_mlp = nn.Sequential(nn.Linear(2*dim_f, 2*dim_f, bias=False),
                                           nn.ReLU(),
                                           nn.Linear(2*dim_f, 4, bias=False))

    def forward(self, feat_f0_unfold, feat_f1_unfold, data):
        """
        Args:
            feat_f0_unfold (torch.Tensor): [M, WW, C]
            feat_f1_unfold (torch.Tensor): [M, WW, C]
            data (dict)
        Update:
            data (dict):{
                'expec_f' (torch.Tensor): [M, 3],
                'mkpts0_f' (torch.Tensor): [M, 2],
                'mkpts1_f' (torch.Tensor): [M, 2]}
        """

        feat_f0 = self.fine_proj(feat_f0_unfold)
        feat_f1 = self.fine_proj(feat_f1_unfold)

        M, WW, C = feat_f0.shape
        W_f = self.W_f

        # corner case: if no coarse matches found
        if M == 0:
            assert self.training == False, "M is always >0, when training, see coarse_matching.py"
            # logger.warning('No matches found in coarse-level.')
            data.update({
                'mkpts0_f': data['mkpts0_c'],
                'mkpts1_f': data['mkpts1_c'],
                'mconf_f': torch.zeros(0, device=feat_f0_unfold.device),
                # 'mkpts0_f_train': data['mkpts0_c'],
                # 'mkpts1_f_train': data['mkpts1_c'],
                # 'conf_matrix_fine': torch.zeros(1, W_f*W_f, W_f*W_f, device=feat_f0.device)
            })
            return

        # normalize
        feat_f0, feat_f1 = map(lambda feat: feat / feat.shape[-1]**.5,
                               [feat_f0, feat_f1])
        sim_matrix = torch.einsum("nlc,nsc->nls", feat_f0,
                                      feat_f1) / self.temperature

        conf_matrix_fine = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)
        data.update({'conf_matrix_fine': conf_matrix_fine})

        # predict fine-level and sub-pixel matches from conf_matrix
        data.update(**self.get_fine_sub_match(conf_matrix_fine, feat_f0_unfold, feat_f1_unfold, data))

    def get_fine_sub_match(self, conf_matrix_fine, feat_f0_unfold, feat_f1_unfold, data):
        """
        Args:
            conf_matrix_fine (torch.Tensor): [M, WW, WW]
            feat_f0_unfold (torch.Tensor): [M, WW, C]
            feat_f1_unfold (torch.Tensor): [M, WW, C]
            data (dict)
        Update:
            data (dict):{
                'm_bids' (torch.Tensor): [M]
                'expec_f' (torch.Tensor): [M, 3],
                'mkpts0_f' (torch.Tensor): [M, 2],
                'mkpts1_f' (torch.Tensor): [M, 2]}
        """

        with torch.no_grad():
            W_f = self.W_f

            # 1. confidence thresholding
            mask = conf_matrix_fine > self.fine_thr

            if mask.sum() == 0:
                mask[0,0,0] = 1
                conf_matrix_fine[0,0,0] = 1

            if not self.denser:
                # match only the highest confidence
                mask = mask \
                    * (conf_matrix_fine == conf_matrix_fine.amax(dim=[1,2], keepdim=True))
            else:
                # 2. mutual nearest, match all features in fine window
                mask = mask \
                    * (conf_matrix_fine == conf_matrix_fine.max(dim=2, keepdim=True)[0]) \
                    * (conf_matrix_fine == conf_matrix_fine.max(dim=1, keepdim=True)[0])

            # 3. find all valid fine matches
            # this only works when at most one `True` in each row
            mask_v, all_j_ids = mask.max(dim=2)
            b_ids, i_ids = torch.where(mask_v)
            j_ids = all_j_ids[b_ids, i_ids]
            mconf = conf_matrix_fine[b_ids, i_ids, j_ids]

            # 4. update with matches in original image resolution

            # indices from coarse matches
            b_ids_c, i_ids_c, j_ids_c = data['b_ids'], data['i_ids'], data['j_ids']

            # scale (coarse level / fine-level)
            scale_f_c = data['hw0_f'][0] // data['hw0_c'][0]

            # coarse level matches scaled to fine-level (1/2)
            mkpts0_c_scaled_to_f = torch.stack(
            [i_ids_c % data['hw0_c'][1], torch.div(i_ids_c, data['hw0_c'][1], rounding_mode='trunc')],
            dim=1) * scale_f_c

            mkpts1_c_scaled_to_f = torch.stack(
                [j_ids_c % data['hw1_c'][1], torch.div(j_ids_c, data['hw1_c'][1], rounding_mode='trunc')],
                dim=1) * scale_f_c

            # updated b_ids after second thresholding
            updated_b_ids = b_ids_c[b_ids]

            # scales (image res / fine level)
            scale = data['hw0_i'][0] / data['hw0_f'][0]
            scale0 = scale * data['scale0'][updated_b_ids] if 'scale0' in data else scale
            scale1 = scale * data['scale1'][updated_b_ids] if 'scale1' in data else scale

            # fine-level discrete matches on window coordiantes
            mkpts0_f_window = torch.stack(
            [i_ids % W_f, torch.div(i_ids, W_f, rounding_mode='trunc')],
            dim=1)

            mkpts1_f_window = torch.stack(
            [j_ids % W_f, torch.div(j_ids, W_f, rounding_mode='trunc')],
            dim=1)

        # sub-pixel refinement
        sub_ref = self.subpixel_mlp(torch.cat([feat_f0_unfold[b_ids, i_ids],
                                                     feat_f1_unfold[b_ids, j_ids]], dim=-1))
        sub_ref0, sub_ref1 = torch.chunk(sub_ref, 2, dim=-1)
        sub_ref0 = torch.tanh(sub_ref0) * 0.5
        sub_ref1 = torch.tanh(sub_ref1) * 0.5

        # final sub-pixel matches by (coarse-level + fine-level windowed + sub-pixel refinement)
        mkpts0_f_train = (mkpts0_f_window + mkpts0_c_scaled_to_f[b_ids] - (W_f//2) + sub_ref0) * scale0
        mkpts1_f_train = (mkpts1_f_window + mkpts1_c_scaled_to_f[b_ids] - (W_f//2) + sub_ref1) * scale1
        mkpts0_f = mkpts0_f_train.clone().detach()
        mkpts1_f = mkpts1_f_train.clone().detach()

        # These matches is the current prediction (for visualization)
        sub_pixel_matches = {
            'm_bids': b_ids_c[b_ids[mconf != 0]],  # mconf == 0 => gt matches
            'mkpts0_f': mkpts0_f[mconf != 0],
            'mkpts1_f': mkpts1_f[mconf != 0],
            'mconf_f': mconf[mconf != 0]
        }

        # These matches are used for training
        if not self.inference:
            sub_pixel_matches.update({
                'mkpts0_f_train': mkpts0_f_train[mconf != 0],
                'mkpts1_f_train': mkpts1_f_train[mconf != 0],
            })

        return sub_pixel_matches


# ----------------------------------------------------------------------------------------------------
# upstream src/xoftr/xoftr.py
# ----------------------------------------------------------------------------------------------------

class XoFTR(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Misc
        self.config = config

        # Modules
        self.backbone = ResNet_8_2(config['resnet'])
        self.pos_encoding = PositionEncodingSine(config['coarse']['d_model'])
        self.loftr_coarse = LocalFeatureTransformer(config['coarse'])
        self.coarse_matching = CoarseMatching(config['match_coarse'])
        self.fine_process = FineProcess(config)
        self.fine_matching= FineSubMatching(config)


    def forward(self, data):
        """
        Update:
            data (dict): {
                'image0': (torch.Tensor): (N, 1, H, W)
                'image1': (torch.Tensor): (N, 1, H, W)
                'mask0'(optional) : (torch.Tensor): (N, H, W) '0' indicates a padded position
                'mask1'(optional) : (torch.Tensor): (N, H, W)
            }
        """
        # 1. Local Feature CNN
        data.update({
            'bs': data['image0'].size(0),
            'hw0_i': data['image0'].shape[2:], 'hw1_i': data['image1'].shape[2:]
        })

        eps = 1e-6

        image0_mean = data['image0'].mean(dim=[2,3], keepdim=True)
        image0_std = data['image0'].std(dim=[2,3], keepdim=True)
        image0 = (data['image0'] - image0_mean) / (image0_std + eps)

        image1_mean = data['image1'].mean(dim=[2,3], keepdim=True)
        image1_std = data['image1'].std(dim=[2,3], keepdim=True)
        image1 = (data['image1'] - image1_mean) / (image1_std + eps)

        if data['hw0_i'] == data['hw1_i']:  # faster & better BN convergence
            feats_c, feats_m, feats_f = self.backbone(torch.cat([image0, image1], dim=0))
            (feat_c0, feat_c1) = feats_c.split(data['bs'])
            (feat_m0, feat_m1) = feats_m.split(data['bs'])
            (feat_f0, feat_f1) = feats_f.split(data['bs'])
        else:  # handle different input shapes
            feat_c0, feat_m0, feat_f0 = self.backbone(image0)
            feat_c1, feat_m1, feat_f1 = self.backbone(image1)

        data.update({
            'hw0_c': feat_c0.shape[2:], 'hw1_c': feat_c1.shape[2:],
            'hw0_m': feat_m0.shape[2:], 'hw1_m': feat_m1.shape[2:],
            'hw0_f': feat_f0.shape[2:], 'hw1_f': feat_f1.shape[2:]
        })

        # save coarse features for fine matching
        feat_c0_pre, feat_c1_pre = feat_c0.clone(), feat_c1.clone()

        # 2. coarse-level loftr module
        # add featmap with positional encoding, then flatten it to sequence [N, HW, C]
        feat_c0 = rearrange(self.pos_encoding(feat_c0), 'n c h w -> n (h w) c')
        feat_c1 = rearrange(self.pos_encoding(feat_c1), 'n c h w -> n (h w) c')

        mask_c0 = mask_c1 = None  # mask is useful in training
        if 'mask0' in data:
            mask_c0, mask_c1 = data['mask0'].flatten(-2), data['mask1'].flatten(-2)
        feat_c0, feat_c1 = self.loftr_coarse(feat_c0, feat_c1, mask_c0, mask_c1)

        # 3. match coarse-level
        self.coarse_matching(feat_c0, feat_c1, data, mask_c0=mask_c0, mask_c1=mask_c1)

        # 4. fine-level matching module
        feat_f0_unfold, feat_f1_unfold = self.fine_process(feat_f0, feat_f1,
                                                           feat_m0, feat_m1,
                                                           feat_c0, feat_c1,
                                                           feat_c0_pre, feat_c1_pre,
                                                           data)

        # 5. match fine-level and sub-pixel refinement
        self.fine_matching(feat_f0_unfold, feat_f1_unfold, data)

    def load_state_dict(self, state_dict, *args, **kwargs):
        for k in list(state_dict.keys()):
            if k.startswith('matcher.'):
                state_dict[k.replace('matcher.', '', 1)] = state_dict.pop(k)
        return super().load_state_dict(state_dict, *args, **kwargs)

**Module 4/7:** `src/xoftr_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections.abc import Callable
from pathlib import Path
from typing import Any

import torch
from huggingface_hub import snapshot_download

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-
#: local snapshot directory named by the model key and described by the committed manifest; a
#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / DEFAULT_MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory snapshot, no repository checkout


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_configs: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")

    weight_path = root / MODEL_FILENAME
    if not weight_path.is_file():
        raise RuntimeError(f"Pinned checkpoint is missing {MODEL_FILENAME}")

    unsafe = sorted(
        p.name
        for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS
    )
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")

    manifest_path = root / MANIFEST_NAME
    manifest_verified = False

    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc

        files = manifest.get("files") or []
        if not files:
            raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")

        for entry in files:
            rel_path = entry.get("path")
            if not rel_path:
                continue
            target = root / rel_path
            if not target.is_file():
                raise RuntimeError(f"Manifest file missing: {rel_path}")
            exp_bytes = entry.get("bytes")
            if exp_bytes is not None and target.stat().st_size != exp_bytes:
                raise RuntimeError(
                    f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}"
                )
            exp_sha = entry.get("sha256")
            if exp_sha is not None and _sha256(target) != exp_sha:
                raise RuntimeError(f"SHA-256 mismatch for {rel_path}")

        manifest_verified = True

    size = weight_path.stat().st_size
    if size != MODEL_SIZE_BYTES:
        raise RuntimeError(f"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}")

    digest = _sha256(weight_path)
    if digest != MODEL_SHA256:
        raise RuntimeError(
            f"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}"
        )

    # The network's configuration is carried in code (modeling.default_config); with
    # require_configs there is nothing else to require.

    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing"
        )
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    The identity in the manifest must be the pinned one; every manifest entry is then size- and
    SHA-256-checked by :func:`verify_checkpoint` (the existing verifier, which also asserts the
    weight file's pinned digest and byte count and refuses unsafe formats). Returns
    ``{"path": ..., **manifest}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    _, manifest_verified = verify_checkpoint(
        root, require_configs=True, return_manifest_verified=True
    )
    if not manifest_verified:
        raise RuntimeError(f"manifest at {root} was not verified")  # pragma: no cover
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and
    the small files but git-ignores the weights). Returns the relative paths fetched;
    :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve weights path with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `XOFTR_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/xoftr` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and weights/ contains xoftr_640.safetensors)
    4. Hugging Face Hub snapshot download -> 'hf_hub'
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"

    env_dir = os.environ.get("XOFTR_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"

    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout to resolve
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MODEL_FILENAME).is_file():
            return repo_weights, "repo_offline"

    hub_path = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            allow_patterns=list(ALLOWED_CHECKPOINT_FILES),
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    )
    return hub_path, "hf_hub"


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(
    *, coarse_threshold: float = COARSE_THRESHOLD, fine_threshold: float = FINE_THRESHOLD
) -> Any:
    """The vendored XoFTR network at random initialisation, in inference configuration."""
    pass  # standalone rewrite (build_notebook.py): `from .modeling import XoFTR, default_config` removed — names are kernel globals defined by the carried modules

    return XoFTR(default_config(coarse_thr=coarse_threshold, fine_thr=fine_threshold))


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    return_metadata: bool = False,
    coarse_threshold: float = COARSE_THRESHOLD,
    fine_threshold: float = FINE_THRESHOLD,
) -> tuple[Any, torch.device, Path] | tuple[Any, torch.device, Path, dict[str, Any]]:
    """Acquire, verify, and load the one supported XoFTR checkpoint into the vendored network
    (strict state-dict load from safetensors; no pickle, no Hub code)."""
    from safetensors.torch import load_file

    candidate_path, source = resolve_weights_path(
        weights_path=weights_path,
        cache_dir=cache_dir,
    )

    verified, manifest_verified = verify_checkpoint(
        candidate_path,
        require_configs=True,
        return_manifest_verified=True,
    )
    target_device = _resolve_device(device)

    weight_file = verified / MODEL_FILENAME
    state = load_file(str(weight_file))
    if len(state) != STATE_TENSORS:
        raise RuntimeError(f"{MODEL_FILENAME}: {len(state)} tensors, expected {STATE_TENSORS}")
    model = build_model(coarse_threshold=coarse_threshold, fine_threshold=fine_threshold)
    # the vendored class strips the checkpoint's `matcher.` prefix
    model.load_state_dict(state, strict=True)
    n_params = sum(p.numel() for p in model.parameters())
    if n_params != PARAMETER_COUNT:
        raise RuntimeError(
            f"vendored network has {n_params} parameters, expected {PARAMETER_COUNT}"
        )
    model = model.eval().to(target_device)

    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "manifest_verified": manifest_verified,
        "weight_sha256": _sha256(weight_file),
        "weight_size_bytes": weight_file.stat().st_size,
        "device": str(target_device),
        "state_tensors": len(state),
        "parameters": n_params,
    }

    if return_metadata:
        return model, target_device, verified, metadata
    return model, target_device, verified

**Module 5/7:** `src/xoftr_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .modeling import UPSTREAM_COMMIT, UPSTREAM_REPOSITORY` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = (
    "huggingface-hub",
    "numpy",
    "pillow",
    "safetensors",
    "torch",
)


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_sha256 = MODEL_SHA256
    weight_size = MODEL_SIZE_BYTES
    device_str = None
    resolved_checkpoint_path = None
    adapter = None

    if pipeline is not None:
        if getattr(pipeline, "checkpoint_path", None) is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        checkpoint_source = getattr(pipeline, "checkpoint_source", None)
        manifest_verified = bool(getattr(pipeline, "manifest_verified", False))
        if getattr(pipeline, "weight_sha256", None):
            weight_sha256 = pipeline.weight_sha256
        if getattr(pipeline, "weight_size_bytes", None):
            weight_size = pipeline.weight_size_bytes
        if getattr(pipeline, "device", None) is not None:
            device_str = str(pipeline.device)
        if getattr(pipeline, "adapter", None):
            skip = ("history", "trainable_names")
            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}
    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": {
            "id": MODEL_ID,
            "revision": MODEL_REVISION,
            "weight_file": MODEL_FILENAME,
            "weight_sha256": weight_sha256,
            "weight_size_bytes": weight_size,
            "checkpoint_source": checkpoint_source,
            "checkpoint_path": resolved_checkpoint_path,
            "manifest_verified": manifest_verified,
            "vendored_code": {"repository": UPSTREAM_REPOSITORY, "commit": UPSTREAM_COMMIT},
        },
        "preprocessing": {
            "grey_scale": True,
            "crop_to_multiple_of": DIVISIBLE_BY,
            "side_range_px": [MIN_SIDE, MAX_SIDE],
            "per_image_standardisation": "inside the network (mean / std over the image)",
        },
        "inference": {
            "coarse_threshold": COARSE_THRESHOLD,
            "fine_threshold": FINE_THRESHOLD,
            "match_confidence_semantics": "dual_softmax_and_fine_scores_not_calibrated_probability",
            "geometric_verification": (
                "none in the pipeline; the evaluation's RANSAC-DLT is a metric, not a filter"
            ),
            "device": device_str,
        },
        "adapter": adapter,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> Path:
    record = build_provenance(
        pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime
    )
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return out

**Module 6/7:** `src/xoftr_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Image-pair dataset contract for adapting the matcher: the pinned iNaturalist photograph corpus, seeded
homography pairs with exact references, validation, splitting, BYOD loaders and CSV export.

The images are **real**: 360 CC0-licensed, research-grade iNaturalist photographs of six North American bird
species (the fleet's SigLIP sample; 60 per species, one per observer), pinned here by photo id, byte size and
SHA-256 of the served `medium` JPEG and fetched from the iNaturalist open-data bucket at run time, refused on any
byte-size or SHA-256 mismatch; the repository redistributes none of them, and every record keeps its
observation page and observer login. Each photograph becomes one **pair**: the photograph (long side scaled to
640 px, sides cropped to multiples of 8) and a copy warped by a seeded homography with seeded photometric
changes, so the reference `H` (image0 → image1) is exact and every returned match has a reprojection error.
Two difficulty tiers alternate per photograph: `easy` (corner jitter up to 6 % of the side, rotation ±10°,
scale 0.9–1.1, mild brightness / contrast / noise) and `hard` (jitter up to 18 %, rotation ±35°, scale 0.6–1.4,
strong brightness / contrast / gamma / blur / noise).

A record is ``{id, image0, image1, homography}`` (PIL images or paths, and a 3 × 3 list); `SPECIES` maps the
photograph's species key to its names for provenance.
"""
# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

# standalone rewrite (build_notebook.py): `from .config import DIVISIBLE_BY, MAX_SIDE, MIN_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules

MAX_IMAGE_SIDE = 4096  # pixels; larger uploads are rejected before any decode-to-tensor work
WORKING_LONG_SIDE = 640  # the pairs are built at the checkpoint's training resolution

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = (
    "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)"
)
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = (
    "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
)
CORPUS_BYTES = 39_223_447
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, species, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "song_sparrow-10",
        "song_sparrow",
        637315932,
        349374463,
        "sooji",
        136572,
        "a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033",
        "jpg",
    ),
    (
        "song_sparrow-11",
        "song_sparrow",
        471146686,
        262252507,
        "jeanpaulboerekamps",
        98601,
        "4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b",
        "jpg",
    ),
    (
        "song_sparrow-12",
        "song_sparrow",
        640811426,
        351179648,
        "erikschiff",
        107695,
        "27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b",
        "jpg",
    ),
    (
        "song_sparrow-13",
        "song_sparrow",
        123859010,
        75689904,
        "w_mark_c",
        190819,
        "6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33",
        "jpg",
    ),
    (
        "song_sparrow-14",
        "song_sparrow",
        108520869,
        67204020,
        "dugald",
        52496,
        "e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8",
        "jpg",
    ),
    (
        "song_sparrow-15",
        "song_sparrow",
        63537800,
        40010230,
        "nathanael15",
        51441,
        "5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8",
        "jpg",
    ),
    (
        "song_sparrow-16",
        "song_sparrow",
        435251292,
        244042351,
        "carterdorscht",
        166662,
        "d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a",
        "jpeg",
    ),
    (
        "song_sparrow-17",
        "song_sparrow",
        120710033,
        73898230,
        "tys_rbg",
        126820,
        "515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41",
        "jpg",
    ),
    (
        "song_sparrow-18",
        "song_sparrow",
        349361283,
        198243065,
        "sean579",
        42820,
        "5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074",
        "jpg",
    ),
    (
        "song_sparrow-19",
        "song_sparrow",
        393408927,
        222067370,
        "irenemacaulay_",
        101681,
        "22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c",
        "jpg",
    ),
    (
        "song_sparrow-20",
        "song_sparrow",
        614258628,
        337847585,
        "joy4birds",
        70552,
        "aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67",
        "jpg",
    ),
    (
        "song_sparrow-21",
        "song_sparrow",
        608802621,
        335154467,
        "jamesadney",
        112564,
        "6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513",
        "jpg",
    ),
    (
        "song_sparrow-22",
        "song_sparrow",
        634100352,
        347744524,
        "zorthesosen",
        214044,
        "6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd",
        "jpg",
    ),
    (
        "song_sparrow-23",
        "song_sparrow",
        50065474,
        31954532,
        "truthseqr",
        82521,
        "4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15",
        "jpeg",
    ),
    (
        "song_sparrow-24",
        "song_sparrow",
        8656044,
        6803564,
        "glmory",
        297964,
        "e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5",
        "jpg",
    ),
    (
        "song_sparrow-25",
        "song_sparrow",
        131051149,
        79988590,
        "funvill",
        72572,
        "0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894",
        "jpeg",
    ),
    (
        "song_sparrow-26",
        "song_sparrow",
        12923156,
        9491600,
        "gambolingquail",
        69559,
        "6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d",
        "jpg",
    ),
    (
        "song_sparrow-27",
        "song_sparrow",
        12077500,
        8959545,
        "reuvenm",
        74010,
        "3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593",
        "jpeg",
    ),
    (
        "song_sparrow-28",
        "song_sparrow",
        132730299,
        80955309,
        "steph123456",
        154309,
        "5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270",
        "jpeg",
    ),
    (
        "song_sparrow-29",
        "song_sparrow",
        124840110,
        76316566,
        "terrimewbornagain",
        81046,
        "2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6",
        "jpeg",
    ),
    (
        "song_sparrow-30",
        "song_sparrow",
        130188079,
        79482300,
        "fake_id",
        180462,
        "1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4",
        "jpeg",
    ),
    (
        "song_sparrow-31",
        "song_sparrow",
        136405967,
        83075831,
        "tom_lazar",
        178554,
        "f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9",
        "jpeg",
    ),
    (
        "song_sparrow-32",
        "song_sparrow",
        118885385,
        72867352,
        "ellyne",
        105156,
        "aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673",
        "jpeg",
    ),
    (
        "song_sparrow-33",
        "song_sparrow",
        676467457,
        370293984,
        "sholsenbeck",
        59104,
        "b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16",
        "jpg",
    ),
    (
        "song_sparrow-34",
        "song_sparrow",
        688316087,
        376461614,
        "doublecritch",
        109416,
        "7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1",
        "jpg",
    ),
    (
        "song_sparrow-35",
        "song_sparrow",
        691785499,
        378254663,
        "karuquebec",
        144522,
        "9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc",
        "jpg",
    ),
    (
        "song_sparrow-36",
        "song_sparrow",
        691561091,
        378140427,
        "vaughnshirey",
        87827,
        "a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045",
        "jpg",
    ),
    (
        "song_sparrow-37",
        "song_sparrow",
        582182748,
        321880490,
        "dougbrown",
        44795,
        "465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f",
        "jpg",
    ),
    (
        "song_sparrow-38",
        "song_sparrow",
        704199014,
        384692464,
        "enspring",
        85269,
        "1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043",
        "jpg",
    ),
    (
        "song_sparrow-39",
        "song_sparrow",
        583754170,
        322666295,
        "rociherrera",
        97743,
        "5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737",
        "jpg",
    ),
    (
        "song_sparrow-40",
        "song_sparrow",
        590270305,
        325961498,
        "damienxw",
        133375,
        "a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19",
        "jpg",
    ),
    (
        "song_sparrow-41",
        "song_sparrow",
        423534602,
        237940479,
        "don54",
        25229,
        "69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea",
        "jpeg",
    ),
    (
        "song_sparrow-42",
        "song_sparrow",
        418537114,
        235351206,
        "memoosborne",
        95540,
        "3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa",
        "jpeg",
    ),
    (
        "song_sparrow-43",
        "song_sparrow",
        280724703,
        162358184,
        "shirleymorrison",
        43015,
        "f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2",
        "jpg",
    ),
    (
        "song_sparrow-44",
        "song_sparrow",
        641009256,
        351278414,
        "robinlanark",
        49640,
        "e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06",
        "jpg",
    ),
    (
        "song_sparrow-45",
        "song_sparrow",
        531314531,
        295151228,
        "steveplumb",
        128710,
        "8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600",
        "jpg",
    ),
    (
        "song_sparrow-46",
        "song_sparrow",
        114487462,
        70425556,
        "leahmfulton",
        50091,
        "d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343",
        "jpg",
    ),
    (
        "song_sparrow-47",
        "song_sparrow",
        118804811,
        72823547,
        "heibudas",
        191643,
        "8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f",
        "jpeg",
    ),
    (
        "song_sparrow-48",
        "song_sparrow",
        118498513,
        72671334,
        "seanwashington1",
        80800,
        "940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6",
        "jpeg",
    ),
    (
        "song_sparrow-49",
        "song_sparrow",
        119696805,
        73327300,
        "rambryum",
        68983,
        "65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d",
        "jpeg",
    ),
    (
        "song_sparrow-50",
        "song_sparrow",
        4251416,
        3670620,
        "swells",
        49271,
        "8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d",
        "JPG",
    ),
    (
        "song_sparrow-51",
        "song_sparrow",
        303954368,
        174929167,
        "dianeclark6280",
        157478,
        "14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95",
        "png",
    ),
    (
        "song_sparrow-52",
        "song_sparrow",
        392544542,
        221653008,
        "emckenziewhat",
        138683,
        "7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3",
        "jpeg",
    ),
    (
        "song_sparrow-53",
        "song_sparrow",
        270614122,
        156495271,
        "radkins21",
        92783,
        "7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25",
        "jpg",
    ),
    (
        "song_sparrow-54",
        "song_sparrow",
        139634990,
        84920252,
        "raffib128",
        56293,
        "865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5",
        "jpeg",
    ),
    (
        "song_sparrow-55",
        "song_sparrow",
        25251350,
        16733911,
        "kemper",
        54034,
        "99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e",
        "jpeg",
    ),
    (
        "song_sparrow-56",
        "song_sparrow",
        139129879,
        84639025,
        "wewantashrubbery",
        104736,
        "aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643",
        "jpg",
    ),
    (
        "song_sparrow-57",
        "song_sparrow",
        128242284,
        78367372,
        "stevestevens",
        112226,
        "918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b",
        "jpeg",
    ),
    (
        "song_sparrow-58",
        "song_sparrow",
        478733235,
        266446909,
        "saintaardvark",
        298253,
        "d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21",
        "jpg",
    ),
    (
        "song_sparrow-59",
        "song_sparrow",
        623006598,
        342147975,
        "ryman56",
        60213,
        "73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "chipping_sparrow-10",
        "chipping_sparrow",
        92501489,
        57964052,
        "tniernberger",
        78380,
        "44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5",
        "jpg",
    ),
    (
        "chipping_sparrow-11",
        "chipping_sparrow",
        220257388,
        129611350,
        "gcart043",
        129377,
        "1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361",
        "jpeg",
    ),
    (
        "chipping_sparrow-12",
        "chipping_sparrow",
        478480946,
        266314208,
        "russnamitz",
        61359,
        "8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b",
        "jpeg",
    ),
    (
        "chipping_sparrow-13",
        "chipping_sparrow",
        538480223,
        298914072,
        "hiltonward",
        201271,
        "2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346",
        "jpg",
    ),
    (
        "chipping_sparrow-14",
        "chipping_sparrow",
        58192408,
        36778771,
        "bradenjudson",
        22352,
        "7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75",
        "jpeg",
    ),
    (
        "chipping_sparrow-15",
        "chipping_sparrow",
        80410971,
        50636049,
        "radrat",
        55646,
        "0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667",
        "jpg",
    ),
    (
        "chipping_sparrow-16",
        "chipping_sparrow",
        300376697,
        172991803,
        "matthias55",
        81986,
        "7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b",
        "jpeg",
    ),
    (
        "chipping_sparrow-17",
        "chipping_sparrow",
        370917657,
        209626382,
        "craigmartin",
        101736,
        "feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5",
        "jpg",
    ),
    (
        "chipping_sparrow-18",
        "chipping_sparrow",
        264119989,
        152960401,
        "laurelthrone",
        194107,
        "13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e",
        "jpeg",
    ),
    (
        "chipping_sparrow-19",
        "chipping_sparrow",
        220323755,
        129631264,
        "enspring",
        85687,
        "886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1",
        "jpeg",
    ),
    (
        "chipping_sparrow-20",
        "chipping_sparrow",
        210319996,
        124105091,
        "bunnymom20",
        188328,
        "2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899",
        "jpeg",
    ),
    (
        "chipping_sparrow-21",
        "chipping_sparrow",
        117013164,
        71832078,
        "terrimewbornagain",
        49527,
        "4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486",
        "jpeg",
    ),
    (
        "chipping_sparrow-22",
        "chipping_sparrow",
        660951395,
        362143224,
        "mike_cove",
        148079,
        "41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48",
        "jpg",
    ),
    (
        "chipping_sparrow-23",
        "chipping_sparrow",
        691160217,
        377927837,
        "ben142",
        235922,
        "f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd",
        "jpg",
    ),
    (
        "chipping_sparrow-24",
        "chipping_sparrow",
        707633397,
        386482138,
        "leannestacy",
        238800,
        "3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28",
        "jpg",
    ),
    (
        "chipping_sparrow-25",
        "chipping_sparrow",
        696064427,
        380468188,
        "rdnwoods",
        309547,
        "b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07",
        "jpg",
    ),
    (
        "chipping_sparrow-26",
        "chipping_sparrow",
        702557361,
        383835541,
        "seanwashington1",
        159095,
        "2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91",
        "jpg",
    ),
    (
        "chipping_sparrow-27",
        "chipping_sparrow",
        660337945,
        361814902,
        "kristinpiston",
        158651,
        "a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866",
        "jpg",
    ),
    (
        "chipping_sparrow-28",
        "chipping_sparrow",
        546647394,
        303298450,
        "kzoebel",
        86457,
        "c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df",
        "jpg",
    ),
    (
        "chipping_sparrow-29",
        "chipping_sparrow",
        713495256,
        389515472,
        "lina47741",
        287986,
        "52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3",
        "jpg",
    ),
    (
        "chipping_sparrow-30",
        "chipping_sparrow",
        269951645,
        156136769,
        "conhawn",
        106540,
        "11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c",
        "jpeg",
    ),
    (
        "chipping_sparrow-31",
        "chipping_sparrow",
        535714979,
        297462154,
        "dissectedfrog",
        200498,
        "cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6",
        "jpg",
    ),
    (
        "chipping_sparrow-32",
        "chipping_sparrow",
        557301829,
        308874987,
        "mar_y_sierra_silvestre",
        115174,
        "ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e",
        "jpg",
    ),
    (
        "chipping_sparrow-33",
        "chipping_sparrow",
        652401709,
        357680572,
        "dianeclark6280",
        46782,
        "b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1",
        "jpg",
    ),
    (
        "chipping_sparrow-34",
        "chipping_sparrow",
        657945260,
        360569392,
        "jasonleduc",
        36363,
        "e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea",
        "jpg",
    ),
    (
        "chipping_sparrow-35",
        "chipping_sparrow",
        434553038,
        243681621,
        "stariplativky",
        57489,
        "e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd",
        "jpg",
    ),
    (
        "chipping_sparrow-36",
        "chipping_sparrow",
        207498409,
        122551474,
        "askalotl",
        98446,
        "d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68",
        "jpeg",
    ),
    (
        "chipping_sparrow-37",
        "chipping_sparrow",
        179570776,
        106840941,
        "artemis224",
        138087,
        "74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36",
        "jpg",
    ),
    (
        "chipping_sparrow-38",
        "chipping_sparrow",
        81287617,
        51172999,
        "thyg",
        55659,
        "e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7",
        "jpg",
    ),
    (
        "chipping_sparrow-39",
        "chipping_sparrow",
        343116928,
        195090930,
        "umamimomma",
        52983,
        "a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2",
        "jpg",
    ),
    (
        "chipping_sparrow-40",
        "chipping_sparrow",
        480169680,
        267207764,
        "cvharris",
        144339,
        "d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a",
        "jpeg",
    ),
    (
        "chipping_sparrow-41",
        "chipping_sparrow",
        352953833,
        200088501,
        "aster-asti",
        105786,
        "82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db",
        "jpg",
    ),
    (
        "chipping_sparrow-42",
        "chipping_sparrow",
        622149696,
        341721858,
        "wilderbombyx",
        293652,
        "227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824",
        "jpg",
    ),
    (
        "chipping_sparrow-43",
        "chipping_sparrow",
        503555630,
        280377982,
        "jtdavis05",
        42352,
        "24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372",
        "jpeg",
    ),
    (
        "chipping_sparrow-44",
        "chipping_sparrow",
        509135437,
        283351340,
        "martyndrabik",
        125948,
        "1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c",
        "jpg",
    ),
    (
        "chipping_sparrow-45",
        "chipping_sparrow",
        8076820,
        6402608,
        "msieges",
        166588,
        "6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80",
        "jpeg",
    ),
    (
        "chipping_sparrow-46",
        "chipping_sparrow",
        131917390,
        80486493,
        "tom_lazar",
        137355,
        "10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c",
        "jpeg",
    ),
    (
        "chipping_sparrow-47",
        "chipping_sparrow",
        141348323,
        85865218,
        "j-dehoog",
        71473,
        "d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb",
        "jpeg",
    ),
    (
        "chipping_sparrow-48",
        "chipping_sparrow",
        28879424,
        18850970,
        "andy71",
        194983,
        "ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683",
        "jpeg",
    ),
    (
        "chipping_sparrow-49",
        "chipping_sparrow",
        148831027,
        90084486,
        "ian-wolfe",
        181942,
        "cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49",
        "jpg",
    ),
    (
        "chipping_sparrow-50",
        "chipping_sparrow",
        641370647,
        351455126,
        "hrachski",
        130262,
        "9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908",
        "jpg",
    ),
    (
        "chipping_sparrow-51",
        "chipping_sparrow",
        648268506,
        355462611,
        "potatocthulhu",
        193138,
        "f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa",
        "jpg",
    ),
    (
        "chipping_sparrow-52",
        "chipping_sparrow",
        645412478,
        353758550,
        "ctlqh",
        154844,
        "e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b",
        "jpg",
    ),
    (
        "chipping_sparrow-53",
        "chipping_sparrow",
        121777317,
        74504842,
        "hickl",
        48576,
        "a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c",
        "jpg",
    ),
    (
        "chipping_sparrow-54",
        "chipping_sparrow",
        147992761,
        89606238,
        "benkeen",
        461732,
        "8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d",
        "png",
    ),
    (
        "chipping_sparrow-55",
        "chipping_sparrow",
        27976676,
        18310312,
        "schoenitz",
        66424,
        "64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe",
        "jpg",
    ),
    (
        "chipping_sparrow-56",
        "chipping_sparrow",
        190706448,
        112884695,
        "jbeusmans",
        109296,
        "1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6",
        "jpeg",
    ),
    (
        "chipping_sparrow-57",
        "chipping_sparrow",
        184174664,
        109291439,
        "chrismcv",
        91566,
        "c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c",
        "jpeg",
    ),
    (
        "chipping_sparrow-58",
        "chipping_sparrow",
        72765715,
        45879059,
        "henryfrye",
        146855,
        "f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a",
        "jpg",
    ),
    (
        "chipping_sparrow-59",
        "chipping_sparrow",
        74100878,
        46721618,
        "aredmer",
        132045,
        "ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e",
        "jpg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        260413022,
        150969515,
        "andywilson",
        45958,
        "fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "white_throated_sparrow-10",
        "white_throated_sparrow",
        244260317,
        142471526,
        "deejay",
        75694,
        "d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-11",
        "white_throated_sparrow",
        194732675,
        115373159,
        "wildreturn",
        128989,
        "7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831",
        "jpeg",
    ),
    (
        "white_throated_sparrow-12",
        "white_throated_sparrow",
        341990462,
        194541980,
        "ethologist",
        149260,
        "3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb",
        "jpg",
    ),
    (
        "white_throated_sparrow-13",
        "white_throated_sparrow",
        267625208,
        154862375,
        "laurelthrone",
        148616,
        "0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-14",
        "white_throated_sparrow",
        330539586,
        188793537,
        "efalquet",
        119214,
        "c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-15",
        "white_throated_sparrow",
        193091403,
        114346779,
        "ian-wolfe",
        73751,
        "e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479",
        "jpg",
    ),
    (
        "white_throated_sparrow-16",
        "white_throated_sparrow",
        691050773,
        377873006,
        "dinomariobob",
        167744,
        "cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887",
        "jpg",
    ),
    (
        "white_throated_sparrow-17",
        "white_throated_sparrow",
        440986574,
        246671481,
        "suzannehale",
        141857,
        "4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-18",
        "white_throated_sparrow",
        113217820,
        69733707,
        "kemper",
        97802,
        "4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6",
        "jpeg",
    ),
    (
        "white_throated_sparrow-19",
        "white_throated_sparrow",
        575307217,
        318327478,
        "k-simpkins",
        90416,
        "af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b",
        "jpg",
    ),
    (
        "white_throated_sparrow-20",
        "white_throated_sparrow",
        340420076,
        193765555,
        "don54",
        62714,
        "7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556",
        "jpeg",
    ),
    (
        "white_throated_sparrow-21",
        "white_throated_sparrow",
        562344160,
        311510652,
        "rrfc",
        45842,
        "f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0",
        "jpg",
    ),
    (
        "white_throated_sparrow-22",
        "white_throated_sparrow",
        74598266,
        47039878,
        "ianrwhyte",
        141502,
        "b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-23",
        "white_throated_sparrow",
        588001234,
        324825124,
        "portablecity",
        370187,
        "11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607",
        "jpg",
    ),
    (
        "white_throated_sparrow-24",
        "white_throated_sparrow",
        694103481,
        379467387,
        "memoosborne",
        120656,
        "82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a",
        "jpg",
    ),
    (
        "white_throated_sparrow-25",
        "white_throated_sparrow",
        655749108,
        359408087,
        "russnamitz",
        58449,
        "2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1",
        "jpg",
    ),
    (
        "white_throated_sparrow-26",
        "white_throated_sparrow",
        599110899,
        330395670,
        "jd_flores",
        121343,
        "5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae",
        "jpg",
    ),
    (
        "white_throated_sparrow-27",
        "white_throated_sparrow",
        653888173,
        358443775,
        "toknowtheland",
        144197,
        "20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7",
        "jpg",
    ),
    (
        "white_throated_sparrow-28",
        "white_throated_sparrow",
        295037357,
        170132018,
        "sturuss",
        117077,
        "d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3",
        "jpg",
    ),
    (
        "white_throated_sparrow-29",
        "white_throated_sparrow",
        405042885,
        228270930,
        "carterdorscht",
        122157,
        "f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-30",
        "white_throated_sparrow",
        84971412,
        53425343,
        "davidfbird",
        168442,
        "95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6",
        "jpg",
    ),
    (
        "white_throated_sparrow-31",
        "white_throated_sparrow",
        476335868,
        265210970,
        "aster-asti",
        91852,
        "cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54",
        "jpg",
    ),
    (
        "white_throated_sparrow-32",
        "white_throated_sparrow",
        481170792,
        267732071,
        "kcthetc1",
        46021,
        "2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487",
        "jpeg",
    ),
    (
        "white_throated_sparrow-33",
        "white_throated_sparrow",
        623602892,
        342440773,
        "imperialwoodpecker26",
        112678,
        "66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85",
        "jpg",
    ),
    (
        "white_throated_sparrow-34",
        "white_throated_sparrow",
        622732944,
        342010560,
        "tom_lazar",
        34257,
        "584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146",
        "jpg",
    ),
    (
        "white_throated_sparrow-35",
        "white_throated_sparrow",
        505347090,
        281327144,
        "erikschiff",
        85652,
        "91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847",
        "jpg",
    ),
    (
        "white_throated_sparrow-36",
        "white_throated_sparrow",
        7072170,
        5706496,
        "jmutter",
        171582,
        "de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-37",
        "white_throated_sparrow",
        16303076,
        11431253,
        "robw",
        61348,
        "e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7",
        "jpg",
    ),
    (
        "white_throated_sparrow-38",
        "white_throated_sparrow",
        7149657,
        5761350,
        "bradleysaul",
        108304,
        "adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e",
        "jpg",
    ),
    (
        "white_throated_sparrow-39",
        "white_throated_sparrow",
        27513220,
        17998912,
        "mefisher",
        49587,
        "f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b",
        "jpg",
    ),
    (
        "white_throated_sparrow-40",
        "white_throated_sparrow",
        642121459,
        351843522,
        "jeffcherry",
        53348,
        "c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711",
        "jpg",
    ),
    (
        "white_throated_sparrow-41",
        "white_throated_sparrow",
        642355950,
        351962427,
        "robinlanark",
        54504,
        "c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd",
        "jpg",
    ),
    (
        "white_throated_sparrow-42",
        "white_throated_sparrow",
        638199267,
        349826931,
        "wildaboutwildlife",
        32981,
        "845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f",
        "jpg",
    ),
    (
        "white_throated_sparrow-43",
        "white_throated_sparrow",
        119839965,
        73407944,
        "terrimewbornagain",
        61951,
        "dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757",
        "jpeg",
    ),
    (
        "white_throated_sparrow-44",
        "white_throated_sparrow",
        6236370,
        5078910,
        "ruggedbynature",
        126403,
        "3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123",
        "jpeg",
    ),
    (
        "white_throated_sparrow-45",
        "white_throated_sparrow",
        32221828,
        20882923,
        "christinan",
        132485,
        "b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411",
        "jpeg",
    ),
    (
        "white_throated_sparrow-46",
        "white_throated_sparrow",
        36598789,
        23643589,
        "jessm-c",
        45861,
        "eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-47",
        "white_throated_sparrow",
        163198257,
        98065830,
        "ellyne",
        109261,
        "864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a",
        "jpeg",
    ),
    (
        "white_throated_sparrow-48",
        "white_throated_sparrow",
        162150240,
        97493574,
        "c_burns802",
        188464,
        "0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254",
        "jpeg",
    ),
    (
        "white_throated_sparrow-49",
        "white_throated_sparrow",
        161756778,
        97282492,
        "raffib128",
        67237,
        "77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-50",
        "white_throated_sparrow",
        165146429,
        99107203,
        "philippthompson",
        84518,
        "8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-51",
        "white_throated_sparrow",
        177451999,
        105722646,
        "natepow",
        44983,
        "c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840",
        "jpg",
    ),
    (
        "white_throated_sparrow-52",
        "white_throated_sparrow",
        67630852,
        42591255,
        "schoenitz",
        138616,
        "34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181",
        "jpg",
    ),
    (
        "white_throated_sparrow-53",
        "white_throated_sparrow",
        99106006,
        61898962,
        "glennberry",
        122576,
        "fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28",
        "jpg",
    ),
    (
        "white_throated_sparrow-54",
        "white_throated_sparrow",
        102162606,
        63634844,
        "eug302",
        73329,
        "692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-55",
        "white_throated_sparrow",
        370403033,
        209307218,
        "kuykenwil",
        124312,
        "d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969",
        "jpg",
    ),
    (
        "white_throated_sparrow-56",
        "white_throated_sparrow",
        500790892,
        278893388,
        "quillipede",
        186273,
        "0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85",
        "jpeg",
    ),
    (
        "white_throated_sparrow-57",
        "white_throated_sparrow",
        364738073,
        205369541,
        "sandra1142",
        144424,
        "4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-58",
        "white_throated_sparrow",
        374502828,
        211884837,
        "halliefromcali",
        62476,
        "0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c",
        "jpg",
    ),
    (
        "white_throated_sparrow-59",
        "white_throated_sparrow",
        109241547,
        67591707,
        "blkillin",
        130154,
        "e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014",
        "jpeg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "dark_eyed_junco-10",
        "dark_eyed_junco",
        458710472,
        255914048,
        "joy4birds",
        90973,
        "fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153",
        "jpg",
    ),
    (
        "dark_eyed_junco-11",
        "dark_eyed_junco",
        20784016,
        14046286,
        "gambolingquail",
        93957,
        "a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-12",
        "dark_eyed_junco",
        332842159,
        189933284,
        "jan-konilu",
        110145,
        "d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd",
        "jpeg",
    ),
    (
        "dark_eyed_junco-13",
        "dark_eyed_junco",
        513004508,
        270404136,
        "thevertebratepokedex",
        141252,
        "854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f",
        "jpg",
    ),
    (
        "dark_eyed_junco-14",
        "dark_eyed_junco",
        593499256,
        327583635,
        "orionid",
        108583,
        "7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d",
        "jpg",
    ),
    (
        "dark_eyed_junco-15",
        "dark_eyed_junco",
        256948665,
        149140989,
        "igor322",
        86925,
        "9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a",
        "jpeg",
    ),
    (
        "dark_eyed_junco-16",
        "dark_eyed_junco",
        12533281,
        9255403,
        "braincellsgone",
        40857,
        "be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe",
        "jpg",
    ),
    (
        "dark_eyed_junco-17",
        "dark_eyed_junco",
        12580989,
        9282523,
        "artemis224",
        216253,
        "15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093",
        "jpg",
    ),
    (
        "dark_eyed_junco-18",
        "dark_eyed_junco",
        63590391,
        40041059,
        "bobbyblackmore",
        82853,
        "87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-19",
        "dark_eyed_junco",
        106718005,
        66230973,
        "vicki936",
        41475,
        "e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be",
        "jpeg",
    ),
    (
        "dark_eyed_junco-20",
        "dark_eyed_junco",
        611049594,
        336277421,
        "skylar_schell",
        15425,
        "7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e",
        "jpg",
    ),
    (
        "dark_eyed_junco-21",
        "dark_eyed_junco",
        591147962,
        326403963,
        "toknowtheland",
        106373,
        "07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39",
        "jpg",
    ),
    (
        "dark_eyed_junco-22",
        "dark_eyed_junco",
        459696953,
        256398190,
        "w_mark_c",
        262126,
        "5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c",
        "jpg",
    ),
    (
        "dark_eyed_junco-23",
        "dark_eyed_junco",
        484171775,
        269292238,
        "aschuman",
        59459,
        "89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282",
        "jpg",
    ),
    (
        "dark_eyed_junco-24",
        "dark_eyed_junco",
        469746672,
        261469406,
        "shannon_j",
        74167,
        "ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18",
        "jpg",
    ),
    (
        "dark_eyed_junco-25",
        "dark_eyed_junco",
        357382685,
        202362003,
        "dougbrown",
        56324,
        "70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78",
        "jpeg",
    ),
    (
        "dark_eyed_junco-26",
        "dark_eyed_junco",
        7660371,
        6119391,
        "jeffreyleeisanaturalist",
        108394,
        "0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42",
        "jpg",
    ),
    (
        "dark_eyed_junco-27",
        "dark_eyed_junco",
        437957476,
        245430473,
        "eug302",
        106520,
        "394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-28",
        "dark_eyed_junco",
        339439778,
        193239701,
        "rawcomposition",
        32258,
        "73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a",
        "jpg",
    ),
    (
        "dark_eyed_junco-29",
        "dark_eyed_junco",
        634100291,
        347744522,
        "zorthesosen",
        169264,
        "932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81",
        "jpg",
    ),
    (
        "dark_eyed_junco-30",
        "dark_eyed_junco",
        6198359,
        5055484,
        "glmory",
        108168,
        "d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965",
        "jpg",
    ),
    (
        "dark_eyed_junco-31",
        "dark_eyed_junco",
        6485959,
        5243663,
        "fake_id",
        148254,
        "ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712",
        "jpg",
    ),
    (
        "dark_eyed_junco-32",
        "dark_eyed_junco",
        246318772,
        143610209,
        "wildreturn",
        62090,
        "839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-33",
        "dark_eyed_junco",
        253199424,
        147174876,
        "sean579",
        121660,
        "6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa",
        "jpeg",
    ),
    (
        "dark_eyed_junco-34",
        "dark_eyed_junco",
        244315013,
        142500114,
        "enspring",
        63514,
        "1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f",
        "jpg",
    ),
    (
        "dark_eyed_junco-35",
        "dark_eyed_junco",
        256191686,
        148740024,
        "nartb",
        194532,
        "e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298",
        "jpg",
    ),
    (
        "dark_eyed_junco-36",
        "dark_eyed_junco",
        241013731,
        140770209,
        "nana10",
        196479,
        "fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843",
        "jpeg",
    ),
    (
        "dark_eyed_junco-37",
        "dark_eyed_junco",
        240712071,
        140608440,
        "saintaardvark",
        129995,
        "94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5",
        "jpg",
    ),
    (
        "dark_eyed_junco-38",
        "dark_eyed_junco",
        242909841,
        141750179,
        "calinsdad",
        61773,
        "c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c",
        "jpg",
    ),
    (
        "dark_eyed_junco-39",
        "dark_eyed_junco",
        109385584,
        67667584,
        "ninetoes",
        64185,
        "64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2",
        "jpg",
    ),
    (
        "dark_eyed_junco-40",
        "dark_eyed_junco",
        102027986,
        63561089,
        "michaelnaumoff",
        114346,
        "eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d",
        "jpg",
    ),
    (
        "dark_eyed_junco-41",
        "dark_eyed_junco",
        105652410,
        65622040,
        "don54",
        159856,
        "5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1",
        "jpeg",
    ),
    (
        "dark_eyed_junco-42",
        "dark_eyed_junco",
        267851501,
        154986181,
        "shanebustapbj",
        203657,
        "217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5",
        "jpeg",
    ),
    (
        "dark_eyed_junco-43",
        "dark_eyed_junco",
        268597047,
        155389350,
        "j-dehoog",
        54916,
        "c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f",
        "jpeg",
    ),
    (
        "dark_eyed_junco-44",
        "dark_eyed_junco",
        515881506,
        286951393,
        "jubileej",
        108221,
        "9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c",
        "jpg",
    ),
    (
        "dark_eyed_junco-45",
        "dark_eyed_junco",
        640462423,
        351001488,
        "russnamitz",
        44508,
        "63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65",
        "jpg",
    ),
    (
        "dark_eyed_junco-46",
        "dark_eyed_junco",
        649025513,
        355899297,
        "rocksand2134",
        233215,
        "2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39",
        "jpg",
    ),
    (
        "dark_eyed_junco-47",
        "dark_eyed_junco",
        625101518,
        343192310,
        "jtdavis05",
        106387,
        "4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae",
        "jpg",
    ),
    (
        "dark_eyed_junco-48",
        "dark_eyed_junco",
        509902416,
        283764048,
        "loarie",
        271588,
        "0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda",
        "jpg",
    ),
    (
        "dark_eyed_junco-49",
        "dark_eyed_junco",
        627787858,
        344552039,
        "margohj",
        54717,
        "a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244",
        "jpg",
    ),
    (
        "dark_eyed_junco-50",
        "dark_eyed_junco",
        630444682,
        345899876,
        "robinlanark",
        102578,
        "aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2",
        "jpg",
    ),
    (
        "dark_eyed_junco-51",
        "dark_eyed_junco",
        635784743,
        348597723,
        "lina47741",
        317439,
        "392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54",
        "jpg",
    ),
    (
        "dark_eyed_junco-52",
        "dark_eyed_junco",
        518407567,
        288278997,
        "zzphantom",
        231733,
        "de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71",
        "jpg",
    ),
    (
        "dark_eyed_junco-53",
        "dark_eyed_junco",
        620711072,
        341026776,
        "ethologist",
        186675,
        "13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed",
        "jpg",
    ),
    (
        "dark_eyed_junco-54",
        "dark_eyed_junco",
        638143588,
        349800424,
        "tom_lazar",
        157629,
        "0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23",
        "jpg",
    ),
    (
        "dark_eyed_junco-55",
        "dark_eyed_junco",
        668612910,
        366183670,
        "gendereuphorbia",
        76372,
        "4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb",
        "jpg",
    ),
    (
        "dark_eyed_junco-56",
        "dark_eyed_junco",
        116078128,
        71308227,
        "msieges",
        47042,
        "ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc",
        "jpeg",
    ),
    (
        "dark_eyed_junco-57",
        "dark_eyed_junco",
        117580446,
        72145931,
        "hewwowhy",
        67000,
        "1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618",
        "jpg",
    ),
    (
        "dark_eyed_junco-58",
        "dark_eyed_junco",
        117044036,
        71848977,
        "radrat",
        81035,
        "7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641",
        "jpeg",
    ),
    (
        "dark_eyed_junco-59",
        "dark_eyed_junco",
        11110817,
        8359397,
        "giselle9",
        71741,
        "b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-01",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-02",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-05",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-08",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "house_finch-10",
        "house_finch",
        500744872,
        278868417,
        "pbaff",
        149626,
        "2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0",
        "jpeg",
    ),
    (
        "house_finch-11",
        "house_finch",
        268678834,
        155431721,
        "stevestevens",
        120325,
        "5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79",
        "jpeg",
    ),
    (
        "house_finch-12",
        "house_finch",
        358373136,
        202884575,
        "kcthetc1",
        52329,
        "b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe",
        "jpeg",
    ),
    (
        "house_finch-13",
        "house_finch",
        104227663,
        64793290,
        "verdantpulsar",
        101003,
        "90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5",
        "jpeg",
    ),
    (
        "house_finch-14",
        "house_finch",
        196156834,
        116218899,
        "kgarrett",
        56801,
        "ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0",
        "jpeg",
    ),
    (
        "house_finch-15",
        "house_finch",
        454332148,
        253709711,
        "rlaortiz",
        149985,
        "cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d",
        "jpeg",
    ),
    (
        "house_finch-16",
        "house_finch",
        665287910,
        295246528,
        "dinomariobob",
        74972,
        "5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49",
        "jpg",
    ),
    (
        "house_finch-17",
        "house_finch",
        509969159,
        283798927,
        "damienxw",
        104277,
        "56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805",
        "jpg",
    ),
    (
        "house_finch-18",
        "house_finch",
        168402062,
        100871757,
        "k-simpkins",
        86922,
        "209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d",
        "jpg",
    ),
    (
        "house_finch-19",
        "house_finch",
        661042217,
        362189851,
        "dougbrown",
        56426,
        "ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b",
        "jpg",
    ),
    (
        "house_finch-20",
        "house_finch",
        701862058,
        383472210,
        "dblanco",
        25222,
        "8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f",
        "jpg",
    ),
    (
        "house_finch-21",
        "house_finch",
        709783562,
        387605980,
        "fraskar",
        99555,
        "60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31",
        "jpg",
    ),
    (
        "house_finch-22",
        "house_finch",
        703924789,
        384540425,
        "meshhy",
        110802,
        "2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb",
        "jpg",
    ),
    (
        "house_finch-23",
        "house_finch",
        705516095,
        385380385,
        "m_aniket",
        77797,
        "ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64",
        "jpg",
    ),
    (
        "house_finch-24",
        "house_finch",
        577527110,
        319485540,
        "glmory",
        94407,
        "6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613",
        "jpg",
    ),
    (
        "house_finch-25",
        "house_finch",
        570686433,
        315887836,
        "emilyheaton",
        229263,
        "27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733",
        "jpg",
    ),
    (
        "house_finch-26",
        "house_finch",
        266259557,
        154114419,
        "andywilson",
        47792,
        "b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0",
        "jpeg",
    ),
    (
        "house_finch-27",
        "house_finch",
        269729570,
        156011712,
        "kbkash",
        59048,
        "3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7",
        "jpeg",
    ),
    (
        "house_finch-28",
        "house_finch",
        270277531,
        156183453,
        "aparrot1",
        83424,
        "93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a",
        "jpg",
    ),
    (
        "house_finch-29",
        "house_finch",
        658061500,
        360629741,
        "mike_cove",
        107594,
        "deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372",
        "jpg",
    ),
    (
        "house_finch-30",
        "house_finch",
        545518752,
        302693775,
        "cvharris",
        208917,
        "fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b",
        "jpg",
    ),
    (
        "house_finch-31",
        "house_finch",
        662587006,
        362998939,
        "curran",
        113662,
        "644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63",
        "jpg",
    ),
    (
        "house_finch-32",
        "house_finch",
        665549166,
        364566546,
        "sealgyu",
        130087,
        "e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad",
        "jpg",
    ),
    (
        "house_finch-33",
        "house_finch",
        566433127,
        313653086,
        "fake_id",
        150986,
        "cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f",
        "jpg",
    ),
    (
        "house_finch-34",
        "house_finch",
        404592864,
        228025269,
        "logan_artz",
        78971,
        "d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44",
        "jpeg",
    ),
    (
        "house_finch-35",
        "house_finch",
        423118612,
        237726292,
        "conhawn",
        153347,
        "826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40",
        "jpeg",
    ),
    (
        "house_finch-36",
        "house_finch",
        290080883,
        167451466,
        "vijaybarve",
        89129,
        "f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69",
        "jpeg",
    ),
    (
        "house_finch-37",
        "house_finch",
        296347840,
        170825854,
        "susanaber",
        96563,
        "47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770",
        "jpeg",
    ),
    (
        "house_finch-38",
        "house_finch",
        414288752,
        233136904,
        "wafflemaster135",
        29548,
        "9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125",
        "jpeg",
    ),
    (
        "house_finch-39",
        "house_finch",
        284751592,
        164572867,
        "efalquet",
        80806,
        "b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2",
        "jpeg",
    ),
    (
        "house_finch-40",
        "house_finch",
        205914234,
        121669780,
        "darylnolan",
        44763,
        "c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904",
        "jpeg",
    ),
    (
        "house_finch-41",
        "house_finch",
        96688995,
        60467105,
        "erikschiff",
        145431,
        "dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792",
        "jpg",
    ),
    (
        "house_finch-42",
        "house_finch",
        732358790,
        399184296,
        "katrinamccollough",
        41331,
        "76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d",
        "jpg",
    ),
    (
        "house_finch-43",
        "house_finch",
        83824778,
        52720178,
        "beesbirdsbugs",
        48816,
        "72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a",
        "jpg",
    ),
    (
        "house_finch-44",
        "house_finch",
        349400403,
        198261102,
        "paulgraham",
        105965,
        "b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001",
        "jpeg",
    ),
    (
        "house_finch-45",
        "house_finch",
        484652323,
        269544283,
        "robinlanark",
        29863,
        "a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516",
        "jpeg",
    ),
    (
        "house_finch-46",
        "house_finch",
        341357046,
        194221751,
        "haconra",
        80643,
        "8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85",
        "jpg",
    ),
    (
        "house_finch-47",
        "house_finch",
        622140195,
        341714166,
        "rachel141",
        133938,
        "aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04",
        "jpg",
    ),
    (
        "house_finch-48",
        "house_finch",
        515309527,
        286632225,
        "solunasilver",
        29507,
        "72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05",
        "jpg",
    ),
    (
        "house_finch-49",
        "house_finch",
        514877896,
        286416757,
        "jubileej",
        153667,
        "26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07",
        "jpg",
    ),
    (
        "house_finch-50",
        "house_finch",
        125605349,
        76831730,
        "sarahangulo",
        60174,
        "236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080",
        "jpeg",
    ),
    (
        "house_finch-51",
        "house_finch",
        12881677,
        9467101,
        "artemis224",
        173956,
        "91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9",
        "jpg",
    ),
    (
        "house_finch-52",
        "house_finch",
        124405788,
        76034586,
        "radrat",
        36008,
        "bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6",
        "jpeg",
    ),
    (
        "house_finch-53",
        "house_finch",
        17468099,
        12163814,
        "andy71",
        199644,
        "6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d",
        "jpg",
    ),
    (
        "house_finch-54",
        "house_finch",
        134699271,
        82117365,
        "seanwashington1",
        35334,
        "d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a",
        "jpeg",
    ),
    (
        "house_finch-55",
        "house_finch",
        134287410,
        81854554,
        "omcelroy",
        90394,
        "6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203",
        "jpg",
    ),
    (
        "house_finch-56",
        "house_finch",
        17763955,
        12337763,
        "jennifer510",
        167368,
        "35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b",
        "jpeg",
    ),
    (
        "house_finch-57",
        "house_finch",
        18946521,
        13021113,
        "deejay",
        60576,
        "2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8",
        "jpeg",
    ),
    (
        "house_finch-58",
        "house_finch",
        14590841,
        10492098,
        "silvercat",
        120515,
        "23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a",
        "jpeg",
    ),
    (
        "house_finch-59",
        "house_finch",
        124702263,
        76228853,
        "janeyair",
        162431,
        "5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-10",
        "american_goldfinch",
        143215717,
        86889530,
        "memoosborne",
        129438,
        "ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba",
        "jpeg",
    ),
    (
        "american_goldfinch-11",
        "american_goldfinch",
        403873164,
        227647158,
        "drew_baxter",
        106009,
        "74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49",
        "jpeg",
    ),
    (
        "american_goldfinch-12",
        "american_goldfinch",
        481153019,
        267722510,
        "vicki936",
        255397,
        "3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36",
        "jpg",
    ),
    (
        "american_goldfinch-13",
        "american_goldfinch",
        213311122,
        125763980,
        "hickl",
        24740,
        "0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd",
        "jpeg",
    ),
    (
        "american_goldfinch-14",
        "american_goldfinch",
        417352824,
        234736320,
        "joy4birds",
        81109,
        "357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd",
        "jpeg",
    ),
    (
        "american_goldfinch-15",
        "american_goldfinch",
        72130720,
        45486482,
        "dctphoto",
        178111,
        "17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509",
        "jpeg",
    ),
    (
        "american_goldfinch-16",
        "american_goldfinch",
        45148898,
        28952026,
        "megachile",
        49358,
        "6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252",
        "jpeg",
    ),
    (
        "american_goldfinch-17",
        "american_goldfinch",
        133251099,
        81250513,
        "raffib128",
        128110,
        "863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef",
        "jpeg",
    ),
    (
        "american_goldfinch-18",
        "american_goldfinch",
        155797129,
        93957328,
        "nathanael15",
        74086,
        "7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5",
        "jpg",
    ),
    (
        "american_goldfinch-19",
        "american_goldfinch",
        11400438,
        8535447,
        "akneidel",
        26423,
        "4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b",
        "jpg",
    ),
    (
        "american_goldfinch-20",
        "american_goldfinch",
        55990791,
        35505213,
        "conhawn",
        84915,
        "69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb",
        "jpg",
    ),
    (
        "american_goldfinch-21",
        "american_goldfinch",
        145464774,
        88186208,
        "wildreturn",
        68204,
        "565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592",
        "jpg",
    ),
    (
        "american_goldfinch-22",
        "american_goldfinch",
        637247336,
        289067166,
        "dinomariobob",
        142312,
        "01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd",
        "jpg",
    ),
    (
        "american_goldfinch-23",
        "american_goldfinch",
        460988055,
        257029007,
        "eric112",
        60314,
        "0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9",
        "jpg",
    ),
    (
        "american_goldfinch-24",
        "american_goldfinch",
        59072170,
        37280587,
        "bradenjudson",
        23468,
        "827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42",
        "jpeg",
    ),
    (
        "american_goldfinch-25",
        "american_goldfinch",
        66515260,
        41915252,
        "reuvenm",
        58833,
        "90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7",
        "jpeg",
    ),
    (
        "american_goldfinch-26",
        "american_goldfinch",
        109875643,
        67928198,
        "artemis224",
        217909,
        "99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61",
        "jpg",
    ),
    (
        "american_goldfinch-27",
        "american_goldfinch",
        219478221,
        129171341,
        "cgmayers",
        96993,
        "93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600",
        "jpeg",
    ),
    (
        "american_goldfinch-28",
        "american_goldfinch",
        99106353,
        61899131,
        "thyg",
        50544,
        "0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd",
        "jpg",
    ),
    (
        "american_goldfinch-29",
        "american_goldfinch",
        223110598,
        131133187,
        "marissa3",
        89214,
        "6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e",
        "jpg",
    ),
    (
        "american_goldfinch-30",
        "american_goldfinch",
        104649681,
        65037800,
        "kemper",
        113821,
        "f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461",
        "jpeg",
    ),
    (
        "american_goldfinch-31",
        "american_goldfinch",
        98669936,
        61655828,
        "cloaca_enthusiast",
        31964,
        "3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f",
        "jpg",
    ),
    (
        "american_goldfinch-32",
        "american_goldfinch",
        347798463,
        197465311,
        "aster-asti",
        116039,
        "e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043",
        "jpg",
    ),
    (
        "american_goldfinch-33",
        "american_goldfinch",
        461261879,
        257163676,
        "erikschiff",
        40881,
        "eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826",
        "jpg",
    ),
    (
        "american_goldfinch-34",
        "american_goldfinch",
        343726970,
        195390446,
        "robinlanark",
        53508,
        "8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed",
        "jpeg",
    ),
    (
        "american_goldfinch-35",
        "american_goldfinch",
        352259924,
        199722919,
        "carterdorscht",
        122243,
        "11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6",
        "jpeg",
    ),
    (
        "american_goldfinch-36",
        "american_goldfinch",
        148829913,
        90084462,
        "ian-wolfe",
        172110,
        "4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4",
        "jpg",
    ),
    (
        "american_goldfinch-37",
        "american_goldfinch",
        276388190,
        159958623,
        "dianeclark6280",
        39786,
        "284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db",
        "jpg",
    ),
    (
        "american_goldfinch-38",
        "american_goldfinch",
        409778763,
        230765968,
        "squidtk",
        210988,
        "b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1",
        "jpeg",
    ),
    (
        "american_goldfinch-39",
        "american_goldfinch",
        382773537,
        216432059,
        "rocksand2134",
        195020,
        "ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51",
        "jpeg",
    ),
    (
        "american_goldfinch-40",
        "american_goldfinch",
        290287368,
        167563678,
        "sean579",
        105190,
        "8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014",
        "jpeg",
    ),
    (
        "american_goldfinch-41",
        "american_goldfinch",
        423112648,
        237722733,
        "andrew2285",
        55719,
        "10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754",
        "jpeg",
    ),
    (
        "american_goldfinch-42",
        "american_goldfinch",
        422361237,
        237335568,
        "wildaboutwildlife",
        34855,
        "55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924",
        "jpg",
    ),
    (
        "american_goldfinch-43",
        "american_goldfinch",
        291016386,
        167956837,
        "samallonthesciencemon",
        128110,
        "489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32",
        "jpg",
    ),
    (
        "american_goldfinch-44",
        "american_goldfinch",
        419250127,
        235728207,
        "paulgraham",
        39211,
        "be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9",
        "jpeg",
    ),
    (
        "american_goldfinch-45",
        "american_goldfinch",
        416004544,
        234039610,
        "randv",
        105453,
        "942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f",
        "jpeg",
    ),
    (
        "american_goldfinch-46",
        "american_goldfinch",
        416408092,
        234251817,
        "kmayner",
        143385,
        "55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2",
        "jpeg",
    ),
    (
        "american_goldfinch-47",
        "american_goldfinch",
        116188790,
        71368917,
        "leahmfulton",
        48440,
        "ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc",
        "jpg",
    ),
    (
        "american_goldfinch-48",
        "american_goldfinch",
        118530525,
        72675993,
        "seanwashington1",
        38505,
        "d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738",
        "jpeg",
    ),
    (
        "american_goldfinch-49",
        "american_goldfinch",
        121016576,
        74071097,
        "geenance",
        110180,
        "fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9",
        "jpeg",
    ),
    (
        "american_goldfinch-50",
        "american_goldfinch",
        3279608,
        2872951,
        "joed",
        115964,
        "c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca",
        "jpeg",
    ),
    (
        "american_goldfinch-51",
        "american_goldfinch",
        118140282,
        72458899,
        "melaniegaddy",
        243713,
        "a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8",
        "jpg",
    ),
    (
        "american_goldfinch-52",
        "american_goldfinch",
        7669539,
        6125334,
        "cuihenggang",
        61287,
        "36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9",
        "jpeg",
    ),
    (
        "american_goldfinch-53",
        "american_goldfinch",
        9258865,
        7182023,
        "fake_id",
        151987,
        "131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee",
        "jpeg",
    ),
    (
        "american_goldfinch-54",
        "american_goldfinch",
        12222432,
        9054511,
        "myacadianforest",
        107758,
        "69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb",
        "jpeg",
    ),
    (
        "american_goldfinch-55",
        "american_goldfinch",
        128301260,
        78400716,
        "natepow",
        67416,
        "0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9",
        "jpg",
    ),
    (
        "american_goldfinch-56",
        "american_goldfinch",
        129842959,
        79296314,
        "nartb",
        117002,
        "f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa",
        "jpg",
    ),
    (
        "american_goldfinch-57",
        "american_goldfinch",
        78410307,
        49351195,
        "chrisleearm",
        108066,
        "8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e",
        "jpg",
    ),
    (
        "american_goldfinch-58",
        "american_goldfinch",
        72300928,
        45598176,
        "henrya",
        68593,
        "a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42",
        "jpeg",
    ),
    (
        "american_goldfinch-59",
        "american_goldfinch",
        194724999,
        115367030,
        "vlatassa",
        159105,
        "c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324",
        "jpeg",
    ),
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 36, "validation": 8, "test": 16}  # photographs per species; 6 species -> 216 / 48 / 96 pairs
TIERS = ("easy", "hard")
TIER_PARAMS: dict[str, dict[str, Any]] = {
    "easy": {"jitter": 0.06, "rotation": 10.0, "scale": (0.9, 1.1), "brightness": (0.85, 1.15), "contrast": (0.85, 1.15), "gamma": (1.0, 1.0), "blur": 0.0, "noise": 4.0},
    "hard": {"jitter": 0.18, "rotation": 35.0, "scale": (0.6, 1.4), "brightness": (0.6, 1.4), "contrast": (0.6, 1.4), "gamma": (0.7, 1.4), "blur": 1.2, "noise": 10.0},
}
MIN_RECORDS = 4
MAX_RECORDS = 5_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,
    either case — the bucket key is case-sensitive)."""
    if ext.lower() not in ("jpg", "jpeg", "png"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "dimer-xoftr-tutorial/1.0"}
                )
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, species}` records with their provenance."""
    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "species": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


# --------------------------------------------------------------------------------------------------
# pair synthesis: a seeded homography with an exact reference
# --------------------------------------------------------------------------------------------------


def working_size(size: tuple[int, int], long_side: int = WORKING_LONG_SIDE) -> tuple[int, int]:
    """The size a photograph is brought to: long side `long_side`, both sides multiples of DIVISIBLE_BY."""
    width, height = size
    scale = long_side / max(width, height)
    w = max(DIVISIBLE_BY, int(width * scale) // DIVISIBLE_BY * DIVISIBLE_BY)
    h = max(DIVISIBLE_BY, int(height * scale) // DIVISIBLE_BY * DIVISIBLE_BY)
    return w, h


def prepare_image(image: Image.Image, long_side: int = WORKING_LONG_SIDE) -> Image.Image:
    """Resize to the working size (bicubic, aspect kept up to the multiple-of-8 crop) as RGB."""
    w, h = working_size(image.size, long_side)
    return image.convert("RGB").resize((w, h), Image.BICUBIC)


def sample_homography(size: tuple[int, int], rng: random.Random, params: Mapping[str, Any]) -> np.ndarray:
    """A random homography built from a rotation + scale about the centre followed by corner jitter."""
    width, height = size
    cx, cy = (width - 1) / 2.0, (height - 1) / 2.0
    angle = math.radians(rng.uniform(-params["rotation"], params["rotation"]))
    scale = rng.uniform(*params["scale"])
    cos_a, sin_a = math.cos(angle) * scale, math.sin(angle) * scale
    similarity = np.array(
        [[cos_a, -sin_a, cx - cos_a * cx + sin_a * cy], [sin_a, cos_a, cy - sin_a * cx - cos_a * cy], [0.0, 0.0, 1.0]]
    )
    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])
    jitter = params["jitter"] * min(width, height)
    moved = corners + np.array([[rng.uniform(-jitter, jitter), rng.uniform(-jitter, jitter)] for _ in range(4)])
    pass  # standalone rewrite (build_notebook.py): `from .metrics import dlt_homography` removed — names are kernel globals defined by the carried modules

    perspective = dlt_homography(corners, moved)
    if perspective is None:  # degenerate draw (practically impossible); fall back to the similarity alone
        return similarity
    return perspective @ similarity


def _perspective_coefficients(homography: np.ndarray) -> list[float]:
    """PIL's PERSPECTIVE transform takes the inverse mapping (output pixel -> input pixel), 8 coefficients."""
    inverse = np.linalg.inv(homography)
    inverse = inverse / inverse[2, 2]
    return [float(v) for v in inverse.ravel()[:8]]


def warp_image(image: Image.Image, homography: np.ndarray) -> Image.Image:
    """image1 = image0 warped by `homography` (image0 coords -> image1 coords), same canvas, black outside."""
    return image.transform(image.size, Image.PERSPECTIVE, _perspective_coefficients(homography), Image.BICUBIC)


def photometric(image: Image.Image, rng: random.Random, params: Mapping[str, Any]) -> Image.Image:
    """Seeded brightness / contrast / gamma / blur / Gaussian-noise changes (never geometric)."""
    out = ImageEnhance.Brightness(image).enhance(rng.uniform(*params["brightness"]))
    out = ImageEnhance.Contrast(out).enhance(rng.uniform(*params["contrast"]))
    gamma = rng.uniform(*params["gamma"])
    if params["blur"] > 0:
        out = out.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, params["blur"])))
    array = np.asarray(out, dtype=np.float64) / 255.0
    if gamma != 1.0:
        array = np.power(np.clip(array, 0.0, 1.0), gamma)
    if params["noise"] > 0:
        noise_rng = np.random.default_rng(rng.getrandbits(32))
        array = array + noise_rng.normal(0.0, params["noise"] / 255.0, array.shape)
    return Image.fromarray((np.clip(array, 0.0, 1.0) * 255.0).round().astype(np.uint8))


def make_pair(image: Image.Image, *, seed: int, tier: str = "hard", record_id: str = "pair") -> dict[str, Any]:
    """One `{id, image0, image1, homography, tier}` record from a photograph and a seed."""
    if tier not in TIER_PARAMS:
        raise ValueError(f"tier must be one of {TIERS}")
    params = TIER_PARAMS[tier]
    rng = random.Random(seed)
    image0 = prepare_image(image)
    homography = sample_homography(image0.size, rng, params)
    image1 = photometric(warp_image(image0, homography), rng, params)
    return {
        "id": record_id,
        "image0": image0,
        "image1": image1,
        "homography": homography.tolist(),
        "tier": tier,
        "seed": seed,
    }


def make_pairs(records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, tier: str | None = None) -> list[dict[str, Any]]:
    """One pair per image record (`{id, image, ...}`); tiers alternate easy / hard unless `tier` is fixed."""
    out = []
    for index, record in enumerate(records):
        chosen = tier or TIERS[index % len(TIERS)]
        pair = make_pair(record["image"], seed=seed * 100_003 + index, tier=chosen, record_id=str(record["id"]))
        for key in ("species", "observer", "inat_photo_id", "inat_observation_url", "source_id"):
            if key in record:
                pair[key] = record[key]
        out.append(pair)
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw of photographs per species into train / validation / test, then one pair per
    photograph (tiers alternating within each split). No photograph lands in two splits."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record.get("species", "image")), []).append(dict(record))
    photos: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            photos[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    out: dict[str, list[dict[str, Any]]] = {}
    for offset, name in enumerate(photos):
        rng.shuffle(photos[name])
        relabelled = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(photos[name])]
        out[name] = make_pairs(relabelled, seed=seed + offset)
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


# --------------------------------------------------------------------------------------------------
# validation
# --------------------------------------------------------------------------------------------------


def _open(image: Any, label_name: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    return image.convert("RGB")


def check_homography(value: Any, label_name: str = "homography") -> np.ndarray:
    array = np.asarray(value, dtype=np.float64)
    if array.shape != (3, 3) or not np.all(np.isfinite(array)):
        raise ValueError(f"{label_name}: must be a finite 3x3 matrix")
    if abs(array[2, 2]) < 1e-12:
        raise ValueError(f"{label_name}: H[2, 2] must be non-zero")
    array = array / array[2, 2]
    if abs(np.linalg.det(array)) < 1e-9:
        raise ValueError(f"{label_name}: matrix is singular")
    return array


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image0/image1/homography")
    for key in ("id", "image0", "image1", "homography"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    image0 = _open(record["image0"], label_name + ".image0")
    image1 = _open(record["image1"], label_name + ".image1")
    for which, image in (("image0", image0), ("image1", image1)):
        if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:
            raise ValueError(
                f"{label_name}.{which}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px after preparation; got {image.size}"
            )
    homography = check_homography(record["homography"], label_name + ".homography")
    tier = record.get("tier", "unspecified")
    if not isinstance(tier, str) or not tier:
        raise ValueError(f"{label_name}: tier must be a non-empty string when given")
    item = {"id": rid, "image0": image0, "image1": image1, "homography": homography.tolist(), "tier": tier}
    for key in ("source_id", "seed", "species", "observer", "inat_photo_id", "inat_observation_url"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    """Structural validation of a pair dataset; raises ValueError before any model import. Nothing checks
    that `image1` really is `image0` under `homography` — a wrong reference is scored without complaint."""
    if (
        isinstance(records, Mapping)
        or not isinstance(records, Sequence)
        or isinstance(records, (str, bytes))
    ):
        raise ValueError("records must be a list of {id, image0, image1, homography} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    tiers: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        tiers[item["tier"]] = tiers.get(item["tier"], 0) + 1
        checked.append(item)
    sides = [max(r["image0"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "tiers": dict(sorted(tiers.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same image matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image0"]), image_digest(r["image1"]), r["homography"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no source image (by decoded-pixel digest of image0) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image0"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    """How many observers contributed photos to more than one split (an observation, not an assertion)."""
    seen: dict[str, set[str]] = {}
    for name, records in splits.items():
        for record in records:
            if record.get("observer"):
                seen.setdefault(str(record["observer"]), set()).add(name)
    return {
        "observers": len(seen),
        "in_more_than_one_split": sum(1 for s in seen.values() if len(s) > 1),
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of BYOD image records (`{id, image}`) into train / validation / test after de-duplicating
    images by decoded pixels, then one pair per image."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    seen: set[str] = set()
    unique = []
    for record in records:
        image = _open(record["image"], str(record.get("id")))
        key = image_digest(image)
        if key not in seen:
            seen.add(key)
            unique.append({**record, "image": image})
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    parts = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(parts["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(parts['train'])} training images; at least {MIN_RECORDS} are required")
    out = {}
    for offset, (name, part) in enumerate(parts.items()):
        relabelled = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(part)]
        out[name] = make_pairs(relabelled, seed=seed + offset)
    return out


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image}` records from a directory or a zip of image files (optionally listed in `images.csv`
    with columns `id`, `file`); images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        names = sorted(p.name for p in source.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist() if Path(n).suffix.lower() in (".jpg", ".jpeg", ".png")}
        names = sorted(members)
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding JPEG / PNG image files")
    if not names:
        raise ValueError("BYOD dataset holds no JPEG / PNG image files")
    out = []
    for name in names:
        image = loader(name)
        image.load()
        out.append({"id": re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB")})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pair table of a split (id, source, tier, seed, the homography, provenance)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "source_id", "tier", "seed", "homography", "observer", "inat_observation_url"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "source_id": record.get("source_id", ""),
                    "tier": record.get("tier", ""),
                    "seed": record.get("seed", ""),
                    "homography": json.dumps(record["homography"]),
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out

**Module 7/7:** `src/xoftr_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""XoFTR matching pipeline: the inference contract (`match`), homography-supervised evaluation, a bounded
adaptation of the coarse transformer, and a digest-manifested safetensors adapter."""
# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from io import BytesIO
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .metrics import identity_baseline, matching_metrics, pair_metrics, patch_neighbour_baseline` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import load_components, stage_missing_files, verify_snapshot` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image

# --------------------------------------------------------------------------
# Adaptation contract (E2E): bounded fine-tuning of the coarse transformer's last layers with the
# upstream coarse focal loss, supervised by the pairs' exact homographies.
# --------------------------------------------------------------------------
COARSE_LAYERS = 8  # loftr_coarse: ['self', 'cross'] * 4
DEFAULT_TRAINABLE_COARSE_LAYERS = 2  # the last self + cross pair + the coarse projection (1,378,560 params)
MAX_EVAL_RECORDS = 5_000
MIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.xoftr.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
FOCAL_ALPHA = 0.25  # upstream LOSS.FOCAL_ALPHA / FOCAL_GAMMA / POS_WEIGHT
FOCAL_GAMMA = 2.0
POS_WEIGHT = 1.0

INPUT_SCHEMA: dict[str, Any] = {
    "images": (
        "PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to "
        "grey-scale; remote URLs are refused"
    ),
    "image_size": (
        f"sides in [{MIN_SIDE}, {MAX_SIDE}] px; each image is cropped down to a multiple of {DIVISIBLE_BY} "
        "(the 1/8 coarse grid) before matching and keypoints are reported in the cropped frame"
    ),
    "thresholds": {"coarse": COARSE_THRESHOLD, "fine": FINE_THRESHOLD},
    "output": "kpts0 / kpts1 (M, 2) float32 pixel coordinates (x, y) and confidence (M,) in (0, 1]",
    "validation": (
        "size and decodability only. Nothing checks that the two images show the same scene: any two "
        "images are matched, and a pair with no overlap still returns whatever passes the thresholds"
    ),
}


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, bytes):
        image = Image.open(BytesIO(value))
        image.load()
        return image.convert("RGB")
    if isinstance(value, str | Path):
        text = str(value)
        if text.lower().startswith(("http://", "https://")):
            raise ValueError("remote image URLs are not accepted; pass a local path, bytes or a PIL image")
        path = Path(text)
        if not path.is_file():
            raise ValueError(f"image file not found: {path}")
        image = Image.open(path)
        image.load()
        return image.convert("RGB")
    raise ValueError("image must be a local path, bytes or a PIL.Image.Image")


def _check_size(image: Image.Image, what: str) -> None:
    if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:
        raise ValueError(f"{what}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px; got {image.size}")


def _to_tensor(image: Image.Image) -> torch.Tensor:
    """Grey-scale float tensor (1, 1, H, W) in [0, 1], sides cropped to multiples of DIVISIBLE_BY."""
    width, height = image.size
    w, h = width // DIVISIBLE_BY * DIVISIBLE_BY, height // DIVISIBLE_BY * DIVISIBLE_BY
    grey = image.convert("L").crop((0, 0, w, h))
    array = np.asarray(grey, dtype=np.float32) / 255.0
    return torch.from_numpy(array)[None, None]


def validate_inputs(
    image0: ImageInput, image1: ImageInput, *, names: Sequence[str] | None = None
) -> dict[str, Any]:
    """Validation stage: exactly the checks `match` applies, reported as an input manifest before the model runs."""
    findings: list[dict[str, Any]] = []
    observations = []
    labels = list(names) if names else ["image0", "image1"]
    for label, value in zip(labels, (image0, image1), strict=True):
        image = _coerce_image(value)
        _check_size(image, label)
        width, height = image.size
        observations.append(
            {
                "name": label,
                "size": [width, height],
                "cropped_to": [width // DIVISIBLE_BY * DIVISIBLE_BY, height // DIVISIBLE_BY * DIVISIBLE_BY],
                "mode": "grey-scale after conversion",
            }
        )
    return {"schema": INPUT_SCHEMA, "images": observations, "findings": findings, "verdict": "accepted"}


class XoFTRPipeline:
    def __init__(
        self,
        model: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
    ) -> None:
        self.model = model
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes
        self.adapter: dict[str, Any] | None = None
        if hasattr(self.model, "parameters"):
            for param in self.model.parameters():
                param.requires_grad_(False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        coarse_threshold: float = COARSE_THRESHOLD,
        fine_threshold: float = FINE_THRESHOLD,
    ) -> XoFTRPipeline:
        """Load the one supported checkpoint into the vendored network.

        ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``: absent manifest
        entries are staged with :func:`stage_missing_files` (only when ``allow_download=True``), the directory
        is verified against the manifest and the pinned digest by :func:`verify_snapshot`, and
        :func:`load_components` loads it as an explicit path (nothing goes through ``snapshot_download``).
        """
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            verify_snapshot(weights_dir)
            weights_path = weights_dir
        model, target_device, _, metadata = load_components(
            device=device,
            cache_dir=cache_dir,
            weights_path=weights_path,
            return_metadata=True,
            coarse_threshold=coarse_threshold,
            fine_threshold=fine_threshold,
        )
        return cls(
            model,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
        )

    # ------------------------------------------------------------------ inference contract

    def match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:
        """Dense-to-sparse matches between two images: `{kpts0, kpts1, confidence, size0, size1}` with
        keypoints as (M, 2) float32 (x, y) pixel coordinates in each image's cropped frame."""
        img0, img1 = _coerce_image(image0), _coerce_image(image1)
        _check_size(img0, "image0")
        _check_size(img1, "image1")
        t0, t1 = _to_tensor(img0).to(self.device), _to_tensor(img1).to(self.device)
        batch = {"image0": t0, "image1": t1}
        with torch.inference_mode():
            self.model(batch)
        kpts0 = batch["mkpts0_f"].detach().cpu().numpy().astype(np.float32)
        kpts1 = batch["mkpts1_f"].detach().cpu().numpy().astype(np.float32)
        conf = batch["mconf_f"].detach().cpu().numpy().astype(np.float32)
        return {
            "kpts0": kpts0.reshape(-1, 2),
            "kpts1": kpts1.reshape(-1, 2),
            "confidence": conf.reshape(-1),
            "size0": [int(t0.shape[-1]), int(t0.shape[-2])],
            "size1": [int(t1.shape[-1]), int(t1.shape[-2])],
            "coarse_matches": int(batch["mkpts0_c"].shape[0]),
        }

    # ------------------------------------------------------------------ evaluation

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        matcher: Callable[[Image.Image, Image.Image], Mapping[str, Any]] | None = None,
    ) -> dict[str, Any]:
        """Homography-supervised scoring of a validated pair dataset with `metrics.matching_metrics`; `matcher`
        substitutes a baseline for the model (same record structure, same scoring)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        rows = []
        for record in checked:
            result = (
                matcher(record["image0"], record["image1"])
                if matcher is not None
                else self.match(record["image0"], record["image1"])
            )
            size = tuple(result.get("size0", record["image0"].size))
            row = pair_metrics(result, np.asarray(record["homography"]), (int(size[0]), int(size[1])))
            row.update({"id": record["id"], "tier": record["tier"]})
            rows.append(row)
        out = matching_metrics(rows)
        tiers = sorted({r["tier"] for r in rows})
        out["by_tier"] = {
            tier: {
                k: v
                for k, v in matching_metrics([r for r in rows if r["tier"] == tier]).items()
                if k != "definitions"
            }
            for tier in tiers
        }
        out.update(
            {
                "per_pair": [{k: v for k, v in r.items() if k != "inlier_errors"} for r in rows],
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "matcher": "model" if matcher is None else "baseline",
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return out

    def evaluate_baselines(self, records: Sequence[Mapping[str, Any]]) -> dict[str, dict[str, Any]]:
        """The two non-neural references scored exactly as the model is."""
        out = {}
        for name, fn in (("identity", identity_baseline), ("patch_neighbour", patch_neighbour_baseline)):
            result = self.evaluate(records, matcher=fn)
            result["baseline"] = name
            out[name] = result
        return out

    # ------------------------------------------------------------------ adaptation

    def _coarse_forward(self, t0: torch.Tensor, t1: torch.Tensor) -> tuple[torch.Tensor, tuple[int, int], tuple[int, int]]:
        """The upstream forward up to the coarse similarity matrix (backbone → positional encoding → coarse
        transformer → coarse projection), with gradients; returns sim / temperature and the coarse grids."""
        model = self.model
        eps = 1e-6
        image0 = (t0 - t0.mean(dim=[2, 3], keepdim=True)) / (t0.std(dim=[2, 3], keepdim=True) + eps)
        image1 = (t1 - t1.mean(dim=[2, 3], keepdim=True)) / (t1.std(dim=[2, 3], keepdim=True) + eps)
        feat_c0, _m0, _f0 = model.backbone(image0)
        feat_c1, _m1, _f1 = model.backbone(image1)
        hw0 = (int(feat_c0.shape[2]), int(feat_c0.shape[3]))
        hw1 = (int(feat_c1.shape[2]), int(feat_c1.shape[3]))
        feat_c0 = model.pos_encoding(feat_c0).flatten(2).permute(0, 2, 1)
        feat_c1 = model.pos_encoding(feat_c1).flatten(2).permute(0, 2, 1)
        feat_c0, feat_c1 = model.loftr_coarse(feat_c0, feat_c1, None, None)
        feat_c0 = model.coarse_matching.final_proj(feat_c0)
        feat_c1 = model.coarse_matching.final_proj(feat_c1)
        feat_c0, feat_c1 = feat_c0 / feat_c0.shape[-1] ** 0.5, feat_c1 / feat_c1.shape[-1] ** 0.5
        sim = torch.einsum("nlc,nsc->nls", feat_c0, feat_c1) / model.coarse_matching.temperature
        return sim, hw0, hw1

    @staticmethod
    def coarse_ground_truth(
        homography: np.ndarray, hw0: tuple[int, int], hw1: tuple[int, int], scale: int = DIVISIBLE_BY
    ) -> torch.Tensor:
        """The upstream coarse supervision for a homography: every coarse cell of image0 (its top-left pixel)
        warped into image1 and rounded to the nearest cell, and the reverse, both marked positive (the union
        rule of `spvs_coarse`); out-of-bounds cells are dropped. Returns (1, h0·w0, h1·w1) with 0 / 1 entries."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import warp_points` removed — names are kernel globals defined by the carried modules

        h0, w0 = hw0
        h1, w1 = hw1
        gt = torch.zeros(1, h0 * w0, h1 * w1)
        ys, xs = np.meshgrid(np.arange(h0), np.arange(w0), indexing="ij")
        pts0 = np.stack([xs.ravel(), ys.ravel()], axis=1) * float(scale)
        warped = np.rint(warp_points(pts0, homography) / scale).astype(np.int64)
        ok = (warped[:, 0] >= 0) & (warped[:, 0] < w1) & (warped[:, 1] >= 0) & (warped[:, 1] < h1)
        i_ids = np.nonzero(ok)[0]
        j_ids = warped[ok, 0] + warped[ok, 1] * w1
        gt[0, i_ids, j_ids] = 1.0
        ys1, xs1 = np.meshgrid(np.arange(h1), np.arange(w1), indexing="ij")
        pts1 = np.stack([xs1.ravel(), ys1.ravel()], axis=1) * float(scale)
        back = np.rint(warp_points(pts1, np.linalg.inv(homography)) / scale).astype(np.int64)
        ok1 = (back[:, 0] >= 0) & (back[:, 0] < w0) & (back[:, 1] >= 0) & (back[:, 1] < h0)
        j1 = np.nonzero(ok1)[0]
        i1 = back[ok1, 0] + back[ok1, 1] * w0
        gt[0, i1, j1] = 1.0
        gt[0, 0, 0] = 0.0
        return gt

    @staticmethod
    def coarse_loss(sim: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
        """Upstream `compute_coarse_loss`: the focal term on the positive cells of both softmax directions."""
        conf01 = torch.softmax(sim, 2).clamp(1e-6, 1 - 1e-6)
        conf10 = torch.softmax(sim, 1).clamp(1e-6, 1 - 1e-6)
        pos = gt > 0
        if not bool(pos.any()):
            return sim.sum() * 0.0
        loss = -FOCAL_ALPHA * (1 - conf01[pos]) ** FOCAL_GAMMA * conf01[pos].log()
        loss = loss - FOCAL_ALPHA * (1 - conf10[pos]) ** FOCAL_GAMMA * conf10[pos].log()
        return POS_WEIGHT * loss.mean()

    def _trainable_names(self, trainable_coarse_layers: int) -> list[str]:
        if (
            isinstance(trainable_coarse_layers, bool)
            or not isinstance(trainable_coarse_layers, int)
            or not 1 <= trainable_coarse_layers <= COARSE_LAYERS
        ):
            raise ValueError(f"trainable_coarse_layers must be an int in 1..{COARSE_LAYERS}")
        first = COARSE_LAYERS - trainable_coarse_layers
        prefixes = tuple(f"loftr_coarse.layers.{k}." for k in range(first, COARSE_LAYERS)) + (
            "coarse_matching.final_proj.",
        )
        return [name for name, _p in self.model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 5e-5,
        batch_size: int = 4,
        trainable_coarse_layers: int = DEFAULT_TRAINABLE_COARSE_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the coarse matcher on validated pairs.

        Only the last `trainable_coarse_layers` layers of the coarse transformer (`loftr_coarse`; a self and a
        cross layer by default) and the coarse projection (`coarse_matching.final_proj`) train — 1,378,560 of
        11,091,722 parameters; the backbone, the positional encoding and the whole fine level stay frozen. Each
        pair runs the upstream forward to the coarse similarity matrix and is scored with the upstream coarse
        focal loss against the homography's coarse ground truth; `batch_size` pairs are accumulated per AdamW
        step (pairs have different sizes, so they are not stacked), gradients are clipped at 1.0, the order is
        seeded, no scheduler. Epoch 0 records the frozen model's validation metrics; the epoch with the highest
        validation precision at 3 px is kept (ties broken by homography accuracy at 3 px). Transactional: any
        failure restores the base tensors."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        names = self._trainable_names(trainable_coarse_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        model = self.model
        torch.manual_seed(seed)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        tensors = [(_to_tensor(r["image0"]), _to_tensor(r["image1"]), np.asarray(r["homography"])) for r in train_checked]

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            result = self.evaluate(val_checked)
            return {k: result[k] for k in ("precision_3px", "homography_acc_3px", "inliers_per_pair", "matches_per_pair", "n")}

        def key(entry: dict[str, Any]) -> tuple[float, float]:
            return (entry["val"]["precision_3px"], entry["val"]["homography_acc_3px"]) if entry["val"] else (-math.inf, -math.inf)

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_key = key(entry)
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                model.backbone.eval()  # frozen BatchNorm statistics
                order = torch.randperm(len(tensors), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    optimiser.zero_grad(set_to_none=True)
                    chunk = order[start : start + batch_size]
                    total = 0.0
                    for i in chunk:
                        t0, t1, homography = tensors[i]
                        sim, hw0, hw1 = self._coarse_forward(t0.to(self.device), t1.to(self.device))
                        gt = self.coarse_ground_truth(homography, hw0, hw1).to(self.device)
                        loss = self.coarse_loss(sim, gt) / len(chunk)
                        loss.backward()
                        total += float(loss.detach())
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(total)
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                if not entry["val"] or key(entry) > best_key:
                    best_key = key(entry)
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_coarse_layers": trainable_coarse_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation precision at 3 px (ties: homography accuracy at 3 px)"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ------------------------------------------------------------------ artifacts

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted coarse-matcher tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": DEFAULT_MODEL_KEY,
                "weight_file": MODEL_FILENAME,
                "weight_sha256": MODEL_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256_file(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(f"artifact format_version {manifest.get('format_version')!r} != {ARTIFACT_FORMAT_VERSION!r}")
        base = manifest.get("base_model") or {}
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (MODEL_ID, MODEL_REVISION, MODEL_SHA256):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", MODEL_FILENAME) != MODEL_FILENAME:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1 or files[0].get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must list exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / ARTIFACT_WEIGHTS_NAME).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        layers = (manifest.get("adapter") or {}).get("trainable_coarse_layers")
        expected = sorted(self._trainable_names(layers))
        if sorted(manifest.get("tensors") or []) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        entry = files[0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith(("loftr_coarse.", "coarse_matching.")):
                raise ValueError(f"artifact tensor {key} is not an adaptable coarse-matcher tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | torch.device | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XoFTRPipeline:
        """A fresh pipeline from the pinned base with an adapter overlaid."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def load_pipeline(**kwargs: Any) -> XoFTRPipeline:
    return XoFTRPipeline.from_pretrained(**kwargs)


def evaluation_report(result: Mapping[str, Any], reference: Mapping[str, Any] | None = None, *, sample_kind: str = "synthetic") -> dict[str, Any]:
    """Evaluation stage for the drawn-shape sanity pair: a machine-readable report even when nothing is
    measurable. With a `reference` homography and image size the report carries the pair metrics with the
    verdict `sample-sanity`; without it the verdict is `not-measurable`."""
    base: dict[str, Any] = {
        "task": "detector-free image matching (coarse-to-fine, sub-pixel refined)",
        "score_semantics": (
            "match confidences are dual-softmax / fine-level scores in (0, 1], not calibrated probabilities "
            "that a match is correct; the decision rule is the upstream coarse / fine thresholds; no "
            "geometric verification ships"
        ),
        "sample_kind": sample_kind,
        "n_matches": int(len(np.asarray(result.get("kpts0", [])).reshape(-1, 2))),
    }
    if not reference:
        return {**base, "metrics": [], "verdict": "not-measurable"}
    row = pair_metrics(result, np.asarray(reference["homography"]), tuple(reference["size"]))
    metrics = [
        {"id": "precision_3px", "value": row["precision_3px"], "definition": "fraction of matches under 3 px reprojection error"},
        {"id": "n_inliers", "value": row["n_inliers"], "definition": "matches under 3 px"},
        {"id": "corner_error_px", "value": row["corner_error_px"], "definition": "mean corner displacement of the RANSAC-DLT homography vs the reference"},
    ]
    return {**base, "metrics": metrics, "verdict": "sample-sanity"}


__all__ = [
    "ARTIFACT_FORMAT",
    "COARSE_LAYERS",
    "DEFAULT_TRAINABLE_COARSE_LAYERS",
    "INPUT_SCHEMA",
    "MAX_EVAL_RECORDS",
    "MIN_SCORED_RECORDS",
    "PARAMETER_COUNT",
    "XoFTRPipeline",
    "evaluation_report",
    "load_pipeline",
    "validate_inputs",
]

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `d8ee7d89be3c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `XoFTRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "xoftr",
  "modelId": "vismatch/xoftr",
  "revision": "d8ee7d89be3c9e5c157db3886db1c0f0e038b321",
  "files": [
    {
      "path": "README.md",
      "bytes": 166,
      "sha256": "97e4f408247d1f95334caa95b992bbab2081aff8ba069063c9c069359acd4bb4"
    },
    {
      "path": "vismatch.yaml",
      "bytes": 114,
      "sha256": "64c2401cb5a86b410b2d669c41d4cbdbe97cd9d685a98dbf3619568bf6cbb8f8"
    },
    {
      "path": "xoftr_640.safetensors",
      "bytes": 44419304,
      "sha256": "4d5ed62e8b41f862ecc5c660e31f1c450402966623d6a28e85acf7fbd794cc69"
    }
  ],
  "totalBytes": 44419584
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = XoFTRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. iNaturalist photographs, homography pairs and split

`fetch_corpus` returns the 360 pinned photographs from the cache under `weights/inat-birds/` or the iNaturalist open-data bucket — every cached file is re-hashed and every fetched file refused on any byte-size or SHA-256 mismatch — and `read_corpus` decodes them into image records with their observation page, observer and species. `build_sample_dataset` draws a seeded stratified split of photographs per species (36 / 8 / 16 → 216 / 48 / 96) and turns each photograph into one pair: the photograph at 640 px on the long side (`image0`) and a copy warped by a seeded homography with seeded photometric changes (`image1`), tiers alternating `easy` / `hard`, the reference `H` recorded in the record. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) is shared, `observer_overlap` reports how many observers contributed to more than one split (an observation about the draw, not an assertion), and the training split's pair table is written to `outputs/xoftr_image_matching_train.csv`.

Look for: 360 photographs, three splits with both tiers, three digests, one pair shown with its reference corners, and four refusal probes — a duplicate id, an image over the side ceiling, a singular homography and a dataset too small to split — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod_images': len(records)}
else:
    corpus_files = fetch_corpus(cache_dir='weights/inat-birds')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_RELEASE} ({CORPUS_LICENSE}); one seeded homography pair per photograph'
    raw_rows = {'photographs': len(corpus), 'bytes': sum(len(v) for v in corpus_files.values()), 'observers': len({r['observer'] for r in corpus})}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/xoftr_image_matching_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'observer_overlap': observer_overlap(splits), 'tiers': {k: v for k, v in TIER_PARAMS.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'tiers': manifest['tiers'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
corners = np.array([[0.0, 0.0], [example['image0'].width - 1.0, 0.0], [example['image0'].width - 1.0, example['image0'].height - 1.0], [0.0, example['image0'].height - 1.0]])
print({'example': {'id': example['id'], 'tier': example['tier'], 'size': list(example['image0'].size), 'corners_under_H': np.round(warp_points(corners, np.asarray(example['homography'])), 1).tolist(), 'observation': example.get('inat_observation_url')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'image over the side ceiling': [{**train_records[0], 'image0': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'singular homography': [{**train_records[0], 'homography': [[1, 0, 0], [1, 0, 0], [0, 0, 1]]}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Match through the inference contract

The inference contract is exercised on a drawn pair: a 256 × 192 synthetic scene — a red square, a green circle and a blue triangle on a light background with a faint grid, rendered in code exactly as the repository's `examples/sample-data/generate_samples.py` renders it and digest-asserted against `SHA256SUMS` — and its copy under a fixed reference homography, a different image family from the photographs and a pair the matcher will be asked to match again after adaptation. `validate_inputs` applies exactly the checks `match` applies and returns an input manifest; a remote URL is validated too and its rejection recorded as a finding. `match` returns `kpts0`, `kpts1` and one confidence per correspondence, **ordered as the fine stage emits them**; the per-pair `evaluation_report` on a drawing is `sample-sanity` — plumbing evidence, not a measurement; whether the matcher is *right* is what Section 6 measures on 96 photograph pairs.

In [ ]:
SAMPLE_DIGESTS = {  # examples/sample-data/SHA256SUMS
    'shapes_scene.ppm': '3420b1d3755a5bf9bc90803fa50f3fc7bbeca5720bd6d75748ad79ac00908239',
}
SHAPES_H = np.array([[0.96, 0.05, 12.0], [-0.04, 1.03, -7.0], [2e-5, -1e-5, 1.0]])
WIDTH, HEIGHT, BACKGROUND = 256, 192, (245, 245, 245)


def scene_pixel(x, y):
    if 40 <= x < 104 and 48 <= y < 112:
        return (220, 40, 40)
    if (x - 168) ** 2 + (y - 80) ** 2 <= 34 ** 2:
        return (40, 170, 75)
    if 128 <= y < 176 and abs(x - 120) <= (y - 128) // 2:
        return (40, 90, 220)
    if x % 32 == 0 or y % 32 == 0:
        return (200, 200, 200)
    return BACKGROUND


def render_scene():
    # The repository's generate_samples.py rendering: ASCII P3, 24 values per line.
    lines = ['P3', f'{WIDTH} {HEIGHT}', '255']
    for y in range(HEIGHT):
        row = []
        for x in range(WIDTH):
            row.extend(str(v) for v in scene_pixel(x, y))
        for start in range(0, len(row), 24):
            lines.append(' '.join(row[start : start + 24]))
    return '\n'.join(lines) + '\n'


Path('outputs/sample-data').mkdir(parents=True, exist_ok=True)
scene_path = Path('outputs/sample-data/shapes_scene.ppm')
scene_path.write_bytes(render_scene().encode('ascii'))
scene_digest = hashlib.sha256(scene_path.read_bytes()).hexdigest()
if scene_digest != SAMPLE_DIGESTS['shapes_scene.ppm']:
    raise ValueError(f'Synthetic sample digest mismatch: {scene_digest} != {SAMPLE_DIGESTS["shapes_scene.ppm"]}')
scene0 = Image.open(scene_path).convert('RGB')
scene1 = warp_image(scene0, SHAPES_H)
scene1_path = Path('outputs/sample-data/shapes_scene_warped.png')
scene1.save(scene1_path)
print({'ceilings': {'MIN_SIDE': MIN_SIDE, 'MAX_SIDE': MAX_SIDE, 'DIVISIBLE_BY': DIVISIBLE_BY, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'thresholds': INPUT_SCHEMA['thresholds'], 'device': str(pipe.device)}})
input_manifest = validate_inputs(scene_path, scene1_path, names=[scene_path.name, scene1_path.name])
try:
    validate_inputs('https://example.invalid/not-allowed.png', scene1_path)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'remote-url-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/xoftr_image_matching_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene_sha256': scene_digest[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
t0 = time.perf_counter()
scene_match = pipe.match(scene_path, scene1_path)
scene_errors = reprojection_errors(scene_match['kpts0'], scene_match['kpts1'], SHAPES_H)
checks = {
    'same_length': len(scene_match['kpts0']) == len(scene_match['kpts1']) == len(scene_match['confidence']),
    'coordinates_inside_frames': bool(np.all(scene_match['kpts0'] >= 0) and np.all(scene_match['kpts0'][:, 0] < scene_match['size0'][0]) and np.all(scene_match['kpts0'][:, 1] < scene_match['size0'][1])),
    'confidences_in_unit_interval': bool(np.all((scene_match['confidence'] > 0) & (scene_match['confidence'] <= 1))),
    'some_matches': len(scene_match['kpts0']) > 0,
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
frozen_scene = evaluation_report(scene_match, {'homography': SHAPES_H, 'size': scene_match['size0']}, sample_kind='synthetic')
scene_rows = {'frozen': {'n_matches': int(len(scene_match['kpts0'])), 'precision_3px': float((scene_errors < 3).mean()) if len(scene_errors) else 0.0, 'median_error_px': float(np.median(scene_errors)) if len(scene_errors) else None}}
print({'checks': checks, 'seconds': round(time.perf_counter() - t0, 2), 'frozen_scene': {m['id']: round(m['value'], 3) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict'], 'first_matches': np.round(np.concatenate([scene_match['kpts0'][:3], scene_match['kpts1'][:3]], axis=1), 1).tolist()})

## 6. Baselines and the frozen matcher on the test pairs

Three systems frame the adaptation, each read the same way. The **identity guess** answers that every grid point of image0 is at the same coordinates in image1 — right only where the warp is tiny. The **patch nearest neighbour** answers with the best normalised-cross-correlation 15 × 15 patch within ±48 px — a matcher that knows the images through raw intensities and nothing else. The **frozen model** is scored by `pipe.evaluate`: every returned match against the reference `H`, **precision at 3 px** (also 1 px and 5 px), **matches and inliers per pair**, the **median error** of the inliers, and **homography accuracy at 3 px / 5 px** — the fraction of pairs whose RANSAC-DLT homography from the matches moves the image corners by less than the threshold against the reference, the HPatches-style reading. All three are scored per tier as well. Expect the frozen matcher far above both baselines on every reading: the build record measured precision at 3 px of 0.925 (0.984 easy / 0.866 hard) with 3,393 matches per pair and homography accuracy 0.979, against 0.381 precision for the patch neighbour and 0.007 for the identity guess — and read where it loses: on the hard tier, at 1 px, and in the pairs whose homography still misses.

In [ ]:
METRICS = ('precision_3px', 'precision_1px', 'matches_per_pair', 'inliers_per_pair', 'median_error_px', 'homography_acc_3px', 'homography_acc_5px')


def short(result):
    return {k: (round(result[k], 3) if isinstance(result[k], float) and np.isfinite(result[k]) else result[k]) for k in METRICS}


t0 = time.perf_counter()
baselines = pipe.evaluate_baselines(test_records)
print({'baseline_seconds': round(time.perf_counter() - t0, 1)})
for name, result in baselines.items():
    print({name: short(result), 'by_tier': {tier: short(v) for tier, v in result['by_tier'].items()}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_model_test': short(frozen_test), 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'by_tier_frozen': {tier: short(v) for tier, v in frozen_test['by_tier'].items()}})
print({'definitions': frozen_test['definitions']})
worst = sorted(frozen_test['per_pair'], key=lambda r: r['precision_3px'])[:3]
print({'weakest_pairs_frozen': [{k: r[k] for k in ('id', 'tier', 'n_matches', 'precision_3px', 'corner_error_px')} for r in worst]})
assert frozen_test['precision_3px'] > baselines['patch_neighbour']['precision_3px'] and frozen_test['homography_acc_3px'] > baselines['identity']['homography_acc_3px']

## 7. Bounded fine-tuning of the coarse transformer's last layers

`pipe.adapt` trains only the last `TRAINABLE_COARSE_LAYERS` layers of the coarse transformer (`loftr_coarse`; a self-attention and a cross-attention layer by default) and the coarse projection (`coarse_matching.final_proj`) — 1,378,560 of 11,091,722 parameters; the backbone (with its BatchNorm statistics frozen), the positional encoding and the whole fine stage stay as they are. Each training pair runs the upstream forward to the coarse similarity matrix and is scored with the upstream **coarse focal loss** against the homography's coarse ground truth (every 1/8 cell of image0 warped into image1 and rounded to its nearest cell, and the reverse — the union rule of upstream `spvs_coarse`). AdamW at a fixed learning rate, `BATCH_SIZE` pairs accumulated per step (pairs are not stacked: sizes differ), gradient clipping at 1.0, seeded order, no scheduler. Epoch 0 records the frozen matcher's validation metrics; every epoch is scored on the 48 validation pairs, and the epoch with the highest validation **precision at 3 px** is kept (ties broken by homography accuracy).

Watch the training loss (about 0.72 at epoch 1 and 0.61 at epoch 3 in the build record) and the validation precision: the checkpoint already fits this task, so the selector's job is as much to refuse an epoch that hurts as to keep one that helps. The build record kept epoch 3 of 3 (validation precision at 3 px 0.908 frozen → 0.910); the default is the configuration that gained on the held-out split, and a run that keeps epoch 0 is a valid outcome, not a failure.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_COARSE_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: (round(v, 3) if isinstance(v, float) else v) for k, v in entry['val'].items()})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_coarse_layers=TRAINABLE_COARSE_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test pairs were never used for training or epoch selection, and no photograph appears in two splits. The adapted matcher is scored exactly as the frozen one was in Section 6, the three systems are put side by side on every reading, and the per-tier breakdown is repeated. Read it in this order: **precision at 3 px** first (the metric the epoch was selected on — the build record measured 0.925 → 0.930), then the inlier count and the homography accuracy (0.979 → 0.979), then the tiers, where the easy tier went 0.984 → 0.985 and the hard tier 0.866 → 0.876. The cell asserts only that the adapted matcher is not worse than the frozen one on precision at 3 px by more than a rounding margin — a bounded adaptation of an already-fitted matcher may land flat, and the notebook says so rather than asserting a gain. Ninety-six pairs from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on warped bird photographs says nothing about your scenes until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {metric: {'identity': round(baselines['identity'][metric], 3), 'patch_neighbour': round(baselines['patch_neighbour'][metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS if np.isfinite(frozen_test[metric]) and np.isfinite(adapted_test[metric])}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS if np.isfinite(frozen_test[metric]) and np.isfinite(adapted_test[metric])}
comparison['by_tier'] = {tier: {'frozen': short(frozen_test['by_tier'][tier]), 'adapted': short(adapted_test['by_tier'][tier])} for tier in adapted_test['by_tier']}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': DEFAULT_MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'tiers': TIER_PARAMS,
    'baselines': {name: {k: v for k, v in result.items() if k != 'per_pair'} for name, result in baselines.items()},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'per_pair'},
    'frozen_test_per_pair': frozen_test['per_pair'],
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'per_pair'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'per_pair'},
    'test_metrics_per_pair': adapted_test['per_pair'],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/xoftr_image_matching_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['precision_3px'] >= frozen_test['precision_3px'] - 0.01
print({'report': 'outputs/xoftr_image_matching_evaluation_report.json'})

## 9. Re-match the drawn pair, export the adapter and reload it

The drawn pair from Section 5 is matched again by the adapted model — a drawing, a different image family from the photographs it was tuned on, so this is a small look at what the adaptation did *outside* its corpus (the build record's numbers are in `docs/release-verification.md`; a changed count or precision here is a finding to record, not a failure) — and reported with the per-pair `evaluation_report` (`sample-sanity`). Both match sets are written as JSON.

`pipe.save_artifact` writes the trained tensors — the coarse transformer's last two layers and the coarse projection, about 5.3 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `xoftr_640.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `XoFTRPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the coarse matcher, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical matches on four test pairs (VER4).

In [ ]:
import shutil

adapted_scene_match = pipe.match(scene_path, scene1_path)
adapted_scene_errors = reprojection_errors(adapted_scene_match['kpts0'], adapted_scene_match['kpts1'], SHAPES_H)
adapted_scene = evaluation_report(adapted_scene_match, {'homography': SHAPES_H, 'size': adapted_scene_match['size0']}, sample_kind='synthetic')
scene_rows['adapted'] = {'n_matches': int(len(adapted_scene_match['kpts0'])), 'precision_3px': float((adapted_scene_errors < 3).mean()) if len(adapted_scene_errors) else 0.0, 'median_error_px': float(np.median(adapted_scene_errors)) if len(adapted_scene_errors) else None}
print({'scene': scene_rows, 'scene_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_scene['metrics']}, 'verdict': adapted_scene['verdict']})
with open('outputs/xoftr_image_matching_shapes.json', 'w', encoding='utf-8') as handle:
    json.dump({'reference_homography': SHAPES_H.tolist(), 'frozen': {'kpts0': scene_match['kpts0'].tolist(), 'kpts1': scene_match['kpts1'].tolist(), 'confidence': scene_match['confidence'].tolist()}, 'adapted': {'kpts0': adapted_scene_match['kpts0'].tolist(), 'kpts1': adapted_scene_match['kpts1'].tolist(), 'confidence': adapted_scene_match['confidence'].tolist()}, 'summary': scene_rows}, handle)

artifact_dir = Path('outputs/xoftr_image_matching_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'xoftr_image_matching', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = XoFTRPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
identical = 0
for record in test_records[:4]:
    before, after = pipe.match(record['image0'], record['image1']), reloaded.match(record['image0'], record['image1'])
    identical += int(len(before['kpts0']) == len(after['kpts0']) and np.allclose(before['kpts1'], after['kpts1'], atol=1e-4))
parity = {'identical_pairs': identical, 'of': 4}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_pairs'] == parity['of']

write_provenance('outputs/provenance.json', pipeline=pipe)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'fetched_this_run': fetched, 'weight_file': MODEL_FILENAME, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': MODEL_SHA256, 'vendored_code': {'repository': UPSTREAM_REPOSITORY, 'commit': UPSTREAM_COMMIT}},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': len(SAMPLE_RECORDS), 'tiers': TIER_PARAMS},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'scene': {'sha256': scene_digest, 'reference_homography': SHAPES_H.tolist()}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'dtype': 'float32', 'checkpoint_source': pipe.checkpoint_source},
}
with open('outputs/xoftr_image_matching_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen matcher is already a strong correspondence engine on warped photographs — precision at 3 px of 0.925 with thousands of matches per pair and homography accuracy 0.979, far above a patch nearest neighbour at 0.381 — and a bounded fine-tuning of the coarse transformer's last two layers and projection on 216 pairs moved precision at 3 px from 0.925 to 0.930 (+0.005) with epoch 3 kept, homography accuracy at 3 px 0.979 → 0.979 and 3393 → 3572 matches per pair. That is the claim: the adaptation contract works end to end on a labelled pair set with exact references, and the numbers it produces are read on precision, inlier count and homography accuracy, per tier, against two non-neural baselines and the frozen model rather than in isolation.

The test split is 96 pairs from one seeded draw of one sample, the validation split that picks the epoch is 48, the warps are synthetic (a photograph and its own perspective-warped, re-lit copy — no viewpoint change of a real scene, no occlusion, no thermal modality, which is what the checkpoint was made for), the metrics are reference-based scores (own numpy implementations; none a human judgement), and the build record's own epoch history shows the estimate's fragility: validation precision at 3 px moved 0.908 → 0.906 → 0.908 → 0.910 over epochs 0–3 and homography accuracy 0.958 → 0.979 → 0.979 → 0.979, differences of a few pairs in 48. So a gain here says the contract works, not that the adapted matcher is better on your images, that its confidences are calibrated, or that a match with a high confidence is right — it still returns matches for every pair, and it can be wrong confidently. Fine-tuning on a narrow set can also erode the model elsewhere; the drawn pair re-matched in Section 9 is one image of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the identity guess, the patch neighbour and the frozen matcher's score on *your* pairs are the numbers to read before any adapted one, per tier and on the homography reading. **Leakage:** keep every photograph in one split (the contract de-duplicates by decoded pixels) and split by scene, session or photographer when your images come from few sources — the sample's observer overlap is printed for exactly that reason. **References:** a synthetic warp gives an exact `H`; real pairs need depth, pose or a fitted homography before they can be scored, and a fitted homography is itself an estimate.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot into its vendored network, fetch and digest-verify a real photograph set and build exact-reference pairs from it, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, matching accuracy on real viewpoint changes, other modalities or other cameras, calibration, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_COARSE_LAYERS = 4` and compare the artifact size and the held-out precision; raise `EPOCHS` and watch the validation precision pick the epoch while the training loss keeps falling; change `LEARNING_RATE` to `1e-5` and read a smaller, steadier change; or bring your own photographs through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/xoftr-image-matching-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/xoftr-image-matching-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card (synthetic scene): https://github.com/kurtvalcorza/xoftr-image-matching-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream weights: https://huggingface.co/vismatch/xoftr (the vismatch project's hosting of the XoFTR checkpoints)
- Upstream code: https://github.com/OnderT/XoFTR (vendored as `modeling.py` at the commit recorded in `docs/WEIGHTS.md`)
- XoFTR: Cross-modal Feature Matching Transformer (Tuzcuoğlu, Köksal, Sofu, Kalkan and Alatan, CVPRW 2024): https://arxiv.org/abs/2404.09692
- LoFTR: Detector-Free Local Feature Matching with Transformers (Sun et al., CVPR 2021): https://arxiv.org/abs/2104.00680
- iNaturalist open data (CC0 photographs, each observer's own licence): https://www.inaturalist.org/pages/developers — bucket https://inaturalist-open-data.s3.amazonaws.com/
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)